## CTA chatbot RAG pipeline code

Included in this note book is the entire pipeline created in developing the CTA chatbot. The first section is the web scraping followed by the Mistrial 7B RAG pipeline. We also included our findings/code while testing Llama as well. At the end of the document is the code we developed to create a user interface for our chatbot using Gradio.

Some important notes:\
In order to run this pipeline you will need to provide a ScraperAPI key as well as an OpenAI API key. You might also have to update some of the file paths for the Google Drive mounting and saving parts as well. Everything else should be good to go as is.

## WebScraping the CTA Website for RAG pipeline

### Installing libraries and packages

In [ ]:
!pip install spacy pymupdf requests beautifulsoup4 langchain chromadb curl_cffi tiktoken
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


I will split up the static and the dynamic pages on the CTA website. We will first start scraping data from the static webpages

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import spacy
import fitz  # pymupdf
from urllib.parse import urljoin, urlparse
from collections import Counter

SCRAPER_API_KEY = "2c9b18ed1042a6a0bceef8f45723d331"
BASE_URL = "https://www.transitchicago.com"
SCRAPER_ENDPOINT = "https://api.scraperapi.com"

nlp = spacy.load("en_core_web_sm")

TARGET_PAGES = {
    # --- Travel Information ---
    "travel_info_hub":          "/travel-information/",
    "getting_around":           "/travel-information/getting-around/",
    "rail_station_info":        "/travel-information/rail-station/",
    "howto_finding_your_way":   "/howto/finding-your-way/",
    "howto_paying_fare":        "/howto/pay-for-your-fare/",
    "accessibility_main":       "/accessibility/",
    "accessibility_features":   "/accessibility/features/",
    "accessibility_faq":        "/accessibility/faq/",
    "safety_security":          "/safety/",
    "rules_code_of_conduct":    "/rules/",
    "policies_practices":       "/policies-practices/",
    "bike_and_ride":            "/bikeandride/",
    # --- Fares ---
    "fares_hub":                "/fares/",
    "ventra":                   "/ventra/",
    "ventra_benefits":          "/ventrabenefits/",
    "passes":                   "/passes/",
    "reduced_fare_programs":    "/reduced-fare-programs/",
    "student_reduced_fare":     "/students/",
    "where_to_buy":             "/fares/where-to-buy/",
    # --- Footer additions ---
    "getting_help":             "/getting-help/",
    "ventra_cards_accounts":    "/ventra/cards/",
    "ventra_app":               "/ventra/app/",
    "facts_at_a_glance":        "/facts/",
    "code_of_conduct":          "/code-of-conduct/",
    "people_with_disabilities": "/disabilities/",
    "grade_high_school":        "/students/grade-high-school/",
    "college_students":         "/students/college/",
    "parents_with_children":    "/parents/",
}

NOISE_TAGS = ["script", "style", "nav", "footer", "header", "noscript", "iframe"]

# URLs/patterns to skip when following sublinks
SKIP_URL_PATTERNS = [
    "/alerts/", "/traintracker/", "/bustracker/",   # live/dynamic pages
    "/news/", "/press-releases/",                   # time-sensitive content
    "/jobs/", "/employment/",                       # not rider-facing
    "/governance/", "/board/", "/meetings/",        # admin content
    "javascript:", "mailto:", "tel:",               # non-http links
    "#",                                            # anchor-only links
    "cdn-cgi/"
]


# ── Text Cleaning

def clean_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(NOISE_TAGS):
        tag.decompose()
    raw = soup.get_text(separator="\n", strip=True)
    lines = [line.strip() for line in raw.splitlines() if line.strip()]
    deduped = [lines[0]] if lines else []
    for line in lines[1:]:
        if line != deduped[-1]:
            deduped.append(line)
    return "\n".join(deduped)


# ── Chunking

def chunk_text_by_sentences(text, source, sentences_per_chunk=8, overlap=2):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    if not sentences:
        return []

    chunks = []
    step = sentences_per_chunk - overlap
    for i, start in enumerate(range(0, len(sentences), step)):
        chunk_sentences = sentences[start : start + sentences_per_chunk]
        if not chunk_sentences:
            break
        chunks.append({
            "chunk_id":     f"{source['source_id']}__chunk{i}",
            "source_id":    source["source_id"],
            "source_title": source["source_title"],
            "source_url":   source.get("source_url", ""),
            "source_type":  source.get("source_type", "scraped_web"),
            "text":         " ".join(chunk_sentences),
            "metadata": {
                "scraped_from":       source.get("source_url", ""),
                "sentence_count":     len(chunk_sentences),
                "start_sentence_idx": start,
            },
        })
    return chunks

# ── PDF Extraction

def extract_pdf_text(pdf_bytes: bytes) -> str:
    """Extract and clean text from a PDF using PyMuPDF."""
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        pages_text = []
        for page in doc:
            text = page.get_text("text")
            if text.strip():
                pages_text.append(text.strip())
        doc.close()
        full_text = "\n".join(pages_text)

        # Clean up common PDF artifacts
        lines = [line.strip() for line in full_text.splitlines() if line.strip()]
        deduped = [lines[0]] if lines else []
        for line in lines[1:]:
            if line != deduped[-1]:
                deduped.append(line)
        return "\n".join(deduped)
    except Exception as e:
        print(f"    ⚠ PDF extraction error: {e}")
        return ""


def fetch_pdf(url: str) -> bytes | None:
    """Fetch a PDF via ScraperAPI."""
    try:
        params = {"api_key": SCRAPER_API_KEY, "url": url}
        response = requests.get(SCRAPER_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()
        if b"%PDF" in response.content[:10]:
            return response.content
        # Sometimes ScraperAPI wraps the PDF — try direct fetch as fallback
        direct = requests.get(url, timeout=30)
        if b"%PDF" in direct.content[:10]:
            return direct.content
    except Exception as e:
        print(f"    ⚠ PDF fetch error for {url}: {e}")
    return None


# ── Link Discovery

def should_skip(url: str) -> bool:
    """Return True if the URL should not be followed."""
    for pattern in SKIP_URL_PATTERNS:
        if pattern in url:
            return True
    return False


def discover_links(html: str, parent_url: str) -> tuple[list[str], list[str]]:
    """
    Parse a page's HTML and return:
      - subpage_urls : internal CTA HTML pages worth scraping
      - pdf_urls     : links to PDF/brochure files
    """
    soup = BeautifulSoup(html, "html.parser")
    subpage_urls = []
    pdf_urls = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()
        full_url = urljoin(parent_url, href)
        parsed = urlparse(full_url)

        # Must be on transitchicago.com
        if parsed.netloc and "transitchicago.com" not in parsed.netloc:
            continue

        # Skip unwanted patterns
        if should_skip(full_url):
            continue

        # PDFs and brochures
        if full_url.lower().endswith(".pdf"):
            pdf_urls.append(full_url)

        # Internal HTML subpages
        elif parsed.scheme in ("http", "https") and parsed.path:
            subpage_urls.append(full_url)

    # Deduplicate while preserving order
    subpage_urls = list(dict.fromkeys(subpage_urls))
    pdf_urls     = list(dict.fromkeys(pdf_urls))

    return subpage_urls, pdf_urls


# ── Core Fetch

def fetch_page(url: str, render_js: bool = False) -> str | None:
    """Fetch an HTML page via ScraperAPI. Returns raw HTML or None."""
    try:
        params = {
            "api_key": SCRAPER_API_KEY,
            "url":     url,
            "render":  "true" if render_js else "false",
        }
        response = requests.get(SCRAPER_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()
        if "you have been blocked" in response.text.lower():
            return None
        return response.text
    except Exception as e:
        print(f"    ⚠ Fetch error for {url}: {e}")
        return None


# ── Main Scraper

def scrape_all(target_pages: dict) -> tuple[list, list]:
    raw_docs   = []
    all_chunks = []

    # Track everything we've already visited across all pages
    visited_urls = set()
    visited_pdfs = set()

    for name, path in target_pages.items():
        parent_url = BASE_URL + path

        if parent_url in visited_urls:
            continue

        print(f"\n[{name}] {parent_url}")

        # ── Scrape the parent page
        html = fetch_page(parent_url)
        if not html:
            print(f"  ↻ Retrying with JS rendering...")
            html = fetch_page(parent_url, render_js=True)
        if not html:
            print(f"  ✗ Failed to fetch parent page")
            continue

        visited_urls.add(parent_url)
        text = clean_text(html)
        soup = BeautifulSoup(html, "html.parser")
        title = soup.title.string.strip() if soup.title else name

        if len(text) >= 100:
            print(f"  ✓ Parent page: {len(text)} chars")
            raw_docs.append({"id": name, "url": parent_url, "title": title, "text": text})
            all_chunks.extend(chunk_text_by_sentences(text, {
                "source_id":    name,
                "source_title": title,
                "source_url":   parent_url,
                "source_type":  "scraped_web",
            }))

        # ── Discover sublinks and PDFs on this page
        subpage_urls, pdf_urls = discover_links(html, parent_url)
        print(f"  → Found {len(subpage_urls)} subpages, {len(pdf_urls)} PDFs")

        # ── Scrape subpages
        for sub_url in subpage_urls:
            if sub_url in visited_urls:
                continue

            sub_html = fetch_page(sub_url)
            if not sub_html:
                continue

            visited_urls.add(sub_url)
            sub_text = clean_text(sub_html)

            if len(sub_text) < 100:
                continue

            sub_soup  = BeautifulSoup(sub_html, "html.parser")
            sub_title = sub_soup.title.string.strip() if sub_soup.title else sub_url
            sub_id    = f"{name}__{sub_url.replace(BASE_URL, '').strip('/').replace('/', '_')}"

            print(f"    ✓ Subpage: {sub_url} ({len(sub_text)} chars)")
            raw_docs.append({"id": sub_id, "url": sub_url, "title": sub_title, "text": sub_text})
            all_chunks.extend(chunk_text_by_sentences(sub_text, {
                "source_id":    sub_id,
                "source_title": sub_title,
                "source_url":   sub_url,
                "source_type":  "scraped_web",
            }))

            time.sleep(1)

        # ── Extract PDFs
        for pdf_url in pdf_urls:
            if pdf_url in visited_pdfs:
                continue

            pdf_bytes = fetch_pdf(pdf_url)
            if not pdf_bytes:
                continue

            visited_pdfs.add(pdf_url)
            pdf_text = extract_pdf_text(pdf_bytes)

            if len(pdf_text) < 100:
                continue

            pdf_name  = pdf_url.split("/")[-1].replace(".pdf", "").replace("-", "_").replace("%20", "_")
            pdf_id    = f"{name}__pdf__{pdf_name}"
            pdf_title = pdf_name.replace("_", " ").title()

            print(f"    ✓ PDF: {pdf_url} ({len(pdf_text)} chars)")
            raw_docs.append({"id": pdf_id, "url": pdf_url, "title": pdf_title, "text": pdf_text})
            all_chunks.extend(chunk_text_by_sentences(pdf_text, {
                "source_id":    pdf_id,
                "source_title": pdf_title,
                "source_url":   pdf_url,
                "source_type":  "scraped_pdf",
            }))

            time.sleep(1)

        time.sleep(1.5)

    print(f"\n── Summary ──────────────────────────────────────────")
    print(f"  Total URLs visited : {len(visited_urls)}")
    print(f"  Total PDFs scraped : {len(visited_pdfs)}")
    return raw_docs, all_chunks


# ── Run

print("Scraping CTA pages + sublinks + PDFs via ScraperAPI...\n")

raw_docs, all_chunks = scrape_all(TARGET_PAGES)

raw_docs   = [d for d in raw_docs   if len(d["text"]) >= 100]
all_chunks = [c for c in all_chunks if len(c["text"].strip()) > 0]

with open("cta_raw_docs.json", "w", encoding="utf-8") as f:
    json.dump(raw_docs, f, indent=2, ensure_ascii=False)

with open("cta_rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(raw_docs)} raw documents  → cta_raw_docs.json")
print(f"Saved {len(all_chunks)} RAG chunks    → cta_rag_chunks.json")

Scraping CTA pages + sublinks + PDFs via ScraperAPI...


[travel_info_hub] https://www.transitchicago.com/travel-information/
    ⚠ Fetch error for https://www.transitchicago.com/travel-information/: 403 Client Error: Forbidden for url: https://api.scraperapi.com/?api_key=2c9b18ed1042a6a0bceef8f45723d331&url=https%3A%2F%2Fwww.transitchicago.com%2Ftravel-information%2F&render=false
  ↻ Retrying with JS rendering...
    ⚠ Fetch error for https://www.transitchicago.com/travel-information/: 403 Client Error: Forbidden for url: https://api.scraperapi.com/?api_key=2c9b18ed1042a6a0bceef8f45723d331&url=https%3A%2F%2Fwww.transitchicago.com%2Ftravel-information%2F&render=true
  ✗ Failed to fetch parent page

[getting_around] https://www.transitchicago.com/travel-information/getting-around/
    ⚠ Fetch error for https://www.transitchicago.com/travel-information/getting-around/: 403 Client Error: Forbidden for url: https://api.scraperapi.com/?api_key=2c9b18ed1042a6a0bceef8f45723d331&url=https%3A%2

Apparently the CTA has a static file on the non-static/dynamic pages like "Plan a trip", etc. The CTA contains API's that links live data like train tracker and bus tracker potentially to our system but would require periodic refresh. The GTFS static feed is probably the best solution since it is a static page we just need to embed it once and it does offer some of the info from the live/dynamic pages.

In [ ]:
import requests
import zipfile
import io
import csv
import json

GTFS_URL = "https://www.transitchicago.com/downloads/sch_data/google_transit.zip"

def chunk_text(text, source, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i, start in enumerate(range(0, len(words), step)):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id":     f"{source['source_id']}__chunk{i}",
            "source_id":    source["source_id"],
            "source_title": source["source_title"],
            "source_url":   source.get("source_url", ""),
            "source_type":  source["source_type"],
            "text":         " ".join(chunk_words),
            "metadata":     source.get("metadata", {}),
        })
    return chunks

def read_gtfs_file(z, filename):
    """Safely read a GTFS txt file from the zip, return list of row dicts."""
    if filename not in z.namelist():
        print(f"  ⚠ {filename} not found in GTFS zip — skipping")
        return []
    with z.open(filename) as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig"))
        return list(reader)


def load_gtfs_docs(gtfs_url):
    print("  Downloading GTFS feed...")
    response = requests.get(gtfs_url, timeout=30)
    response.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(response.content))

    print(f"  Files in GTFS zip: {z.namelist()}\n")

    all_chunks = []

    # ── routes.txt ────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "routes.txt")
    for row in rows:
        route_name = f"{row.get('route_short_name','')} {row.get('route_long_name','')}".strip()
        text = (
            f"CTA Route: {route_name}\n"
            f"Route ID: {row.get('route_id', 'N/A')}\n"
            f"Type: {'Bus' if row.get('route_type') == '3' else 'Rail'}\n"
            f"Description: {row.get('route_desc', 'N/A')}\n"
            f"Color: #{row.get('route_color', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_route_{row['route_id']}",
            "source_title": f"Route {route_name}",
            "source_url":   "",
            "source_type":  "gtfs_route",
            "metadata":     {"route_id": row.get("route_id"), "route_type": row.get("route_type")},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ routes.txt      → {len(rows)} rows")

    # ── stops.txt ─────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "stops.txt")
    for row in rows:
        wheelchair = {"0": "No info", "1": "Accessible", "2": "Not accessible"}.get(
            row.get("wheelchair_boarding", "0"), "No info"
        )
        text = (
            f"CTA Stop: {row.get('stop_name', 'N/A')}\n"
            f"Stop ID: {row.get('stop_id', 'N/A')}\n"
            f"Description: {row.get('stop_desc', 'N/A')}\n"
            f"Location: lat {row.get('stop_lat', 'N/A')}, lon {row.get('stop_lon', 'N/A')}\n"
            f"Wheelchair accessible: {wheelchair}\n"
            f"Parent station: {row.get('parent_station', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_stop_{row['stop_id']}",
            "source_title": f"Stop: {row.get('stop_name', 'N/A')}",
            "source_url":   "",
            "source_type":  "gtfs_stop",
            "metadata": {
                "stop_id":             row.get("stop_id"),
                "stop_lat":            row.get("stop_lat"),
                "stop_lon":            row.get("stop_lon"),
                "wheelchair_boarding": row.get("wheelchair_boarding"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ stops.txt       → {len(rows)} rows")

    # ── trips.txt ─────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "trips.txt")
    for row in rows:
        direction = {"0": "Outbound", "1": "Inbound"}.get(row.get("direction_id", ""), "N/A")
        text = (
            f"CTA Trip ID: {row.get('trip_id', 'N/A')}\n"
            f"Route ID: {row.get('route_id', 'N/A')}\n"
            f"Service ID: {row.get('service_id', 'N/A')}\n"
            f"Direction: {direction}\n"
            f"Headsign: {row.get('trip_headsign', 'N/A')}\n"
            f"Shape ID: {row.get('shape_id', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_trip_{row['trip_id']}",
            "source_title": f"Trip: {row.get('trip_headsign', row.get('trip_id', 'N/A'))}",
            "source_url":   "",
            "source_type":  "gtfs_trip",
            "metadata": {
                "trip_id":    row.get("trip_id"),
                "route_id":   row.get("route_id"),
                "service_id": row.get("service_id"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ trips.txt       → {len(rows)} rows")

    # ── calendar.txt ──────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "calendar.txt")
    for row in rows:
        days = [d for d in ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
                if row.get(d) == "1"]
        text = (
            f"CTA Service Schedule: {row.get('service_id', 'N/A')}\n"
            f"Operates on: {', '.join(days) if days else 'N/A'}\n"
            f"Valid from: {row.get('start_date', 'N/A')} to {row.get('end_date', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_calendar_{row['service_id']}",
            "source_title": f"Service Schedule: {row.get('service_id', 'N/A')}",
            "source_url":   "",
            "source_type":  "gtfs_calendar",
            "metadata":     {"service_id": row.get("service_id")},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ calendar.txt    → {len(rows)} rows")

    # ── calendar_dates.txt ────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "calendar_dates.txt")
    for row in rows:
        exception = {"1": "Service added", "2": "Service removed"}.get(
            row.get("exception_type", ""), "N/A"
        )
        text = (
            f"CTA Service Exception\n"
            f"Service ID: {row.get('service_id', 'N/A')}\n"
            f"Date: {row.get('date', 'N/A')}\n"
            f"Exception: {exception}"
        )
        source = {
            "source_id":    f"gtfs_caldate_{row.get('service_id')}_{row.get('date')}",
            "source_title": f"Service Exception: {row.get('service_id')} on {row.get('date')}",
            "source_url":   "",
            "source_type":  "gtfs_calendar_dates",
            "metadata": {
                "service_id":     row.get("service_id"),
                "date":           row.get("date"),
                "exception_type": row.get("exception_type"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ calendar_dates.txt → {len(rows)} rows")

    # ── shapes.txt ────────────────────────────────────────────────────────────
    # Group shape points by shape_id into one doc per shape
    rows = read_gtfs_file(z, "shapes.txt")
    shapes = {}
    for row in rows:
        sid = row.get("shape_id")
        shapes.setdefault(sid, []).append(row)

    for shape_id, points in shapes.items():
        # Summarize rather than dumping every lat/lon point
        text = (
            f"CTA Route Shape ID: {shape_id}\n"
            f"Total points: {len(points)}\n"
            f"Start: lat {points[0].get('shape_pt_lat')}, lon {points[0].get('shape_pt_lon')}\n"
            f"End:   lat {points[-1].get('shape_pt_lat')}, lon {points[-1].get('shape_pt_lon')}"
        )
        source = {
            "source_id":    f"gtfs_shape_{shape_id}",
            "source_title": f"Route Shape: {shape_id}",
            "source_url":   "",
            "source_type":  "gtfs_shape",
            "metadata":     {"shape_id": shape_id, "num_points": len(points)},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ shapes.txt      → {len(shapes)} shapes from {len(rows)} points")

    # ── frequencies.txt (if present) ──────────────────────────────────────────
    rows = read_gtfs_file(z, "frequencies.txt")
    for row in rows:
        text = (
            f"CTA Trip Frequency\n"
            f"Trip ID: {row.get('trip_id', 'N/A')}\n"
            f"Start time: {row.get('start_time', 'N/A')}\n"
            f"End time: {row.get('end_time', 'N/A')}\n"
            f"Headway (seconds between trips): {row.get('headway_secs', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_freq_{row.get('trip_id')}_{row.get('start_time','').replace(':','')}",
            "source_title": f"Frequency: Trip {row.get('trip_id')}",
            "source_url":   "",
            "source_type":  "gtfs_frequency",
            "metadata": {
                "trip_id":      row.get("trip_id"),
                "start_time":   row.get("start_time"),
                "end_time":     row.get("end_time"),
                "headway_secs": row.get("headway_secs"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    if rows:
        print(f"  ✓ frequencies.txt → {len(rows)} rows")

    # ── stop_times.txt ────────────────────────────────────────────────────────
    # ⚠ This file can be millions of rows — group by trip_id to keep chunks meaningful
    rows = read_gtfs_file(z, "stop_times.txt")
    trip_stops = {}
    for row in rows:
        trip_stops.setdefault(row.get("trip_id"), []).append(row)

    for trip_id, stops in trip_stops.items():
        first = stops[0]
        last  = stops[-1]
        text = (
            f"CTA Trip Timetable: {trip_id}\n"
            f"Total stops: {len(stops)}\n"
            f"First stop ID: {first.get('stop_id')} at {first.get('departure_time', 'N/A')}\n"
            f"Last stop ID:  {last.get('stop_id')} at {last.get('arrival_time', 'N/A')}\n"
            f"Intermediate stops: {', '.join(s.get('stop_id','') for s in stops[1:-1][:10])}"
            f"{'...' if len(stops) > 12 else ''}"
        )
        source = {
            "source_id":    f"gtfs_stoptimes_{trip_id}",
            "source_title": f"Timetable: Trip {trip_id}",
            "source_url":   "",
            "source_type":  "gtfs_stop_times",
            "metadata":     {"trip_id": trip_id, "num_stops": len(stops)},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ stop_times.txt  → {len(trip_stops)} trips from {len(rows)} rows")

    return all_chunks


# ── Run

print("Loading GTFS data...\n")
gtfs_chunks = load_gtfs_docs(GTFS_URL)

with open("cta_gtfs_chunks.json", "w", encoding="utf-8") as f:
    json.dump(gtfs_chunks, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(gtfs_chunks)} GTFS chunks → cta_gtfs_chunks.json")

# Breakdown by source type
from collections import Counter
counts = Counter(c["source_type"] for c in gtfs_chunks)
print("\nBreakdown by source_type:")
for source_type, count in counts.most_common():
    print(f"  {source_type:<25} {count} chunks")

Loading GTFS data...

  Files in GTFS zip: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'developers_license_agreement.htm', 'frequencies.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'transfers.txt', 'trips.txt']

  ✓ routes.txt      → 131 rows
  ✓ stops.txt       → 11164 rows
  ✓ trips.txt       → 54663 rows
  ✓ calendar.txt    → 128 rows
  ✓ calendar_dates.txt → 85 rows
  ✓ shapes.txt      → 816 shapes from 717324 points
  ✓ stop_times.txt  → 54663 trips from 3115811 rows

✓ Saved 121650 GTFS chunks → cta_gtfs_chunks.json

Breakdown by source_type:
  gtfs_trip                 54663 chunks
  gtfs_stop_times           54663 chunks
  gtfs_stop                 11164 chunks
  gtfs_shape                816 chunks
  gtfs_route                131 chunks
  gtfs_calendar             128 chunks
  gtfs_calendar_dates       85 chunks


Combining the Static webpages with the GTFS files

In [ ]:
import json
import csv
import requests
import zipfile
import io
from google.colab import files # Import files for download

GTFS_URL = "https://www.transitchicago.com/downloads/sch_data/google_transit.zip"

# ── Source 1: Scraped Static Pages ────────────────────────────────────────────

def load_scraped_docs(filepath: str) -> list[dict]:
    """Load cta_raw_docs.json and normalize into unified chunks."""
    with open(filepath, "r", encoding="utf-8") as f:
        raw_docs = json.load(f)

    all_chunks = []
    for doc in raw_docs:
        source = {
            "source_id":    doc["id"],
            "source_title": doc["title"],
            "source_url":   doc["url"],
            "source_type":  "scraped_web",
            "metadata": {
                "scraped_from": doc["url"],
            },
        }
        all_chunks.extend(chunk_text_by_sentences(doc["text"], source))

    print(f"  Scraped pages  → {len(all_chunks)} chunks from {len(raw_docs)} docs")
    return all_chunks

# ── Source 2: GTFS Static Feed ────────────────────────────────────────────────

def load_gtfs_docs(gtfs_url: str) -> list[dict]:
    """Download GTFS zip and convert key files into unified chunks."""
    print("  Downloading GTFS feed...")
    response = requests.get(gtfs_url, timeout=30)
    response.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(response.content))

    all_chunks = []

    # --- routes.txt ---
    with z.open("routes.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            route_name = f"{row['route_short_name']} {row['route_long_name']}".strip()
            text = (
                f"CTA Route: {route_name}\n"
                f"Route ID: {row['route_id']}\n"
                f"Type: {'Bus' if row['route_type'] == '3' else 'Rail'}\n"
                f"Description: {row.get('route_desc', 'N/A')}\n"
                f"Color: #{row.get('route_color', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_route_{row['route_id']}",
                "source_title": f"Route {route_name}",
                "source_url":   "",
                "source_type":  "gtfs_route",
                "metadata": {
                    "route_id":   row["route_id"],
                    "route_type": row["route_type"],
                },
            }
            all_chunks.extend(chunk_text(text, source))

    # --- stops.txt ---
    with z.open("stops.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            text = (
                f"CTA Stop: {row['stop_name']}\n"
                f"Stop ID: {row['stop_id']}\n"
                f"Location: lat {row.get('stop_lat', 'N/A')}, lon {row.get('stop_lon', 'N/A')}\n"
                f"Wheelchair accessible: {row.get('wheelchair_boarding', 'unknown')}\n"
                f"Description: {row.get('stop_desc', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_stop_{row['stop_id']}",
                "source_title": f"Stop: {row['stop_name']}",
                "source_url":   "",
                "source_type":  "gtfs_stop",
                "metadata": {
                    "stop_id":   row["stop_id"],
                    "stop_lat":  row.get("stop_lat"),
                    "stop_lon":  row.get("stop_lon"),
                    "wheelchair_boarding": row.get("wheelchair_boarding"),
                },
            }
            all_chunks.extend(chunk_text(text, source))

    # --- calendar.txt (service days) ---
    with z.open("calendar.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            days = [d for d in ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"] if row.get(d) == "1"]
            text = (
                f"CTA Service Schedule ID: {row['service_id']}\n"
                f"Operates on: {', '.join(days)}\n"
                f"Valid from {row.get('start_date', 'N/A')} to {row.get('end_date', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_calendar_{row['service_id']}",
                "source_title": f"Service Schedule: {row['service_id']}",
                "source_url":   "",
                "source_type":  "gtfs_calendar",
                "metadata":     {"service_id": row["service_id"]},
            }
            all_chunks.extend(chunk_text(text, source))

    print(f"  GTFS feed      → {len(all_chunks)} chunks")
    return all_chunks

# ── Combine & Save ────────────────────────────────────────────────────────────

def combine_and_save(scraped_path: str, output_path: str):
    print("\nLoading sources...")
    scraped_chunks = load_scraped_docs(scraped_path)
    gtfs_chunks    = load_gtfs_docs(GTFS_URL)

    combined = scraped_chunks + gtfs_chunks

    # Sanity check: flag any duplicate chunk_ids
    ids = [c["chunk_id"] for c in combined]
    dupes = [id for id in ids if ids.count(id) > 1]
    if dupes:
        print(f"  ⚠ Duplicate chunk_ids found: {set(dupes)}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Combined dataset saved → {output_path}")
    print(f"  Total chunks : {len(combined)}")
    print(f"  Scraped web  : {len(scraped_chunks)}")
    print(f"  GTFS         : {len(gtfs_chunks)}")

    # Breakdown by source_type
    from collections import Counter
    counts = Counter(c["source_type"] for c in combined)
    print("\n  Breakdown by source_type:")
    for source_type, count in counts.most_common():
        print(f"    {source_type:<20} {count} chunks")

    files.download(output_path) # Add this line to download the file

combine_and_save(
    scraped_path="cta_raw_docs.json",
    output_path="cta_combined_dataset.json",
)


Loading sources...
  Scraped pages  → 0 chunks from 0 docs
  GTFS feed      → 11423 chunks

✓ Combined dataset saved → cta_combined_dataset.json
  Total chunks : 11423
  Scraped web  : 0
  GTFS         : 11423

  Breakdown by source_type:
    gtfs_stop            11164 chunks
    gtfs_route           131 chunks
    gtfs_calendar        128 chunks


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Mistrial 7B Instruct RAG model (our final model)

Installing Dependencies


In [ ]:
!pip install chromadb sentence-transformers langchain langchain-community
!pip install rank_bm25
!pip install transformers accelerate bitsandbytes>=0.46.1
!pip install ragas datasets langchain-openai

Imports and Configurations

In [ ]:
import json
import os
import pickle
import shutil
import zipfile
import numpy as np
import torch
import warnings
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rank_bm25 import BM25Okapi
import chromadb
from google.colab import files, drive
import pandas as pd

warnings.filterwarnings("ignore")

# ── Configuration
DATASET_PATH    = "cta_combined_dataset.json"
EMBEDDINGS_PATH = "cta_embeddings.pkl"
CHROMA_DB_PATH  = "cta_chroma_db"
COLLECTION_NAME = "cta_knowledge_base"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL_NAME  = "mistralai/Mistral-7B-Instruct-v0.2"

print("✓ Configuration loaded")

✓ Configuration loaded


Mounting Drive and Load Datset

In [ ]:
drive.mount('/content/drive')

SOURCE_PATH = '/content/drive/Shareddrives/ML2-Final Project/Data + Embeddings/cta_combined_dataset.json'

if os.path.exists(SOURCE_PATH):
    shutil.copy(SOURCE_PATH, DATASET_PATH)
    print(f"✓ Copied dataset to '{DATASET_PATH}'")
else:
    print(f"✗ File not found at '{SOURCE_PATH}'")

def load_dataset(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    chunks = [c for c in chunks if c.get("text", "").strip()]
    print(f"✓ Loaded {len(chunks)} chunks from {path}")
    return chunks

chunks = load_dataset(DATASET_PATH)
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Copied dataset to 'cta_combined_dataset.json'
✓ Loaded 12186 chunks from cta_combined_dataset.json

Sample chunk:
{
  "chunk_id": "travel_info_hub__chunk0",
  "source_id": "travel_info_hub",
  "source_title": "Travel Information - Travel info - CTA",
  "source_url": "https://www.transitchicago.com/travel-information/",
  "source_type": "scraped_web",
  "text": "Travel Information - Travel info - CTA\nHome\nTravel info\nShare on Facebook\nShare via email\nClick to print\nTravel Information\nGetting around\nService updates\nMore travel info\nQuick links\nSchedules\nFares\nMaps\nAlerts\nTrackers\nVentra\nSystem status snapshot\n\u2018L\u2019 route status\nRed Line\nNormal Service\nBlue Line\nNormal Service\nBrown Line\nNormal Service\nGreen Line\nNormal Service\nOrange Line\nNormal Service\nPink Line\nNormal Service\nPurple Line\nAdded Service\nYellow Line\nAd

Generate or Load Embeddings

In [ ]:
def generate_and_save_embeddings(chunks, model_name, save_path):
    print(f"Loading embedding model: {model_name}...")
    model = SentenceTransformer(model_name)
    texts = [c["text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks...")
    embeddings = model.encode(
        texts,
        batch_size        = 128 if torch.cuda.is_available() else 32,
        show_progress_bar = True,
        convert_to_numpy  = True,
    )
    save_data = {"embeddings": embeddings, "chunks": chunks, "model_name": model_name}
    with open(save_path, "wb") as f:
        pickle.dump(save_data, f)
    print(f"✓ Embeddings saved → {save_path}  (shape: {embeddings.shape})")
    return embeddings


def load_embeddings(save_path):
    with open(save_path, "rb") as f:
        data = pickle.load(f)
    print(f"✓ Loaded embeddings from {save_path}  (shape: {data['embeddings'].shape})")
    return data["embeddings"], data["chunks"], data["model_name"]


def deduplicate_dataset(chunks, embeddings):
    seen_ids         = set()
    clean_chunks     = []
    clean_embeddings = []
    for chunk, embedding in zip(chunks, embeddings):
        cid = chunk.get("chunk_id", "")
        if cid not in seen_ids:
            seen_ids.add(cid)
            clean_chunks.append(chunk)
            clean_embeddings.append(embedding)
    removed = len(chunks) - len(clean_chunks)
    print(f"✓ Deduplicated: {len(chunks)} → {len(clean_chunks)} chunks "
          f"({removed} duplicates removed)")
    return clean_chunks, np.array(clean_embeddings)


if os.path.exists(EMBEDDINGS_PATH):
    print("Found existing embeddings — loading from disk...")
    embeddings, chunks, model_name = load_embeddings(EMBEDDINGS_PATH)
else:
    print("No existing embeddings — generating from scratch...")
    embeddings = generate_and_save_embeddings(chunks, EMBEDDING_MODEL, EMBEDDINGS_PATH)
    model_name = EMBEDDING_MODEL

# Always deduplicate after loading
chunks, embeddings = deduplicate_dataset(chunks, embeddings)

print(f"\n✓ Embeddings ready: {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions")

Found existing embeddings — loading from disk...
✓ Loaded embeddings from cta_embeddings.pkl  (shape: (12186, 384))
✓ Deduplicated: 12186 → 12183 chunks (3 duplicates removed)

✓ Embeddings ready: 12183 chunks × 384 dimensions


Build ChromaDB

In [ ]:
def build_chroma_db(chunks, embeddings, db_path, collection_name):
    print(f"Building ChromaDB at: {db_path}")
    print(f"Total chunks to index: {len(chunks)}")

    if os.path.exists(db_path):
        shutil.rmtree(db_path)
        print(f"  ✓ Removed old ChromaDB directory")

    client     = chromadb.PersistentClient(path=db_path)
    collection = client.create_collection(
        name     = collection_name,
        metadata = {"hnsw:space": "cosine"},
    )

    batch_size = 500
    total      = len(chunks)
    inserted   = 0

    for start in range(0, total, batch_size):
        batch_chunks     = chunks[start : start + batch_size]
        batch_embeddings = embeddings[start : start + batch_size]

        valid_chunks     = []
        valid_embeddings = []
        for c, e in zip(batch_chunks, batch_embeddings):
            if c.get("chunk_id") and c.get("text", "").strip():
                valid_chunks.append(c)
                valid_embeddings.append(e)

        if not valid_chunks:
            print(f"  ⚠ Batch {start//batch_size + 1} — no valid chunks, skipping")
            continue

        try:
            collection.add(
                ids        = [c["chunk_id"] for c in valid_chunks],
                embeddings = [e.tolist() for e in valid_embeddings],
                documents  = [c["text"] for c in valid_chunks],
                metadatas  = [{
                    "source_id":    c.get("source_id", ""),
                    "source_title": c.get("source_title", ""),
                    "source_url":   c.get("source_url", ""),
                    "source_type":  c.get("source_type", ""),
                } for c in valid_chunks],
            )
            inserted += len(valid_chunks)
            print(f"  Batch {start//batch_size + 1:>3} / {-(-total//batch_size):>3} "
                  f"— inserted {inserted:>6} / {total} chunks")

        except Exception as e:
            print(f"  ✗ Batch {start//batch_size + 1} failed: {e}")
            for c, e_vec in zip(valid_chunks, valid_embeddings):
                try:
                    collection.add(
                        ids        = [c["chunk_id"]],
                        embeddings = [e_vec.tolist()],
                        documents  = [c["text"]],
                        metadatas  = [{
                            "source_id":    c.get("source_id", ""),
                            "source_title": c.get("source_title", ""),
                            "source_url":   c.get("source_url", ""),
                            "source_type":  c.get("source_type", ""),
                        }],
                    )
                    inserted += 1
                except Exception as e2:
                    print(f"    ✗ Skipping {c.get('chunk_id','?')}: {e2}")

    final_count = collection.count()
    print(f"\n{'─'*50}")
    print(f"  Chunks in dataset : {total}")
    print(f"  Chunks in ChromaDB: {final_count}")
    print(f"  {'✓ All indexed' if final_count >= total - 3 else '⚠ Some chunks missing'}")
    print(f"{'─'*50}")
    return client, collection


client, collection = build_chroma_db(chunks, embeddings, CHROMA_DB_PATH, COLLECTION_NAME)

# Verify source types
all_meta     = collection.get(limit=collection.count(), include=["metadatas"])
chroma_types = Counter(m.get("source_type", "MISSING") for m in all_meta["metadatas"])
print("\nSource types in ChromaDB:")
for source_type, count in chroma_types.most_common():
    print(f"  {source_type:<30} {count} chunks")

Building ChromaDB at: cta_chroma_db
Total chunks to index: 12183
  ✓ Removed old ChromaDB directory
  Batch   1 /  25 — inserted    500 / 12183 chunks
  Batch   2 /  25 — inserted   1000 / 12183 chunks
  Batch   3 /  25 — inserted   1500 / 12183 chunks
  Batch   4 /  25 — inserted   2000 / 12183 chunks
  Batch   5 /  25 — inserted   2500 / 12183 chunks
  Batch   6 /  25 — inserted   3000 / 12183 chunks
  Batch   7 /  25 — inserted   3500 / 12183 chunks
  Batch   8 /  25 — inserted   4000 / 12183 chunks
  Batch   9 /  25 — inserted   4500 / 12183 chunks
  Batch  10 /  25 — inserted   5000 / 12183 chunks
  Batch  11 /  25 — inserted   5500 / 12183 chunks
  Batch  12 /  25 — inserted   6000 / 12183 chunks
  Batch  13 /  25 — inserted   6500 / 12183 chunks
  Batch  14 /  25 — inserted   7000 / 12183 chunks
  Batch  15 /  25 — inserted   7500 / 12183 chunks
  Batch  16 /  25 — inserted   8000 / 12183 chunks
  Batch  17 /  25 — inserted   8500 / 12183 chunks
  Batch  18 /  25 — inserted   90

Load Embedding Model and Build BM25

In [ ]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print("✓ Embedding model ready")

def build_bm25_index(chunks):
    tokenized = [c["text"].lower().split() for c in chunks]
    return BM25Okapi(tokenized)

bm25_index = build_bm25_index(chunks)
print(f"✓ BM25 index built over {len(chunks)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model ready
✓ BM25 index built over 12183 chunks


Load LLM

In [ ]:
print(f"Loading {LLM_MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

print(f"✓ {LLM_MODEL_NAME} loaded")

NameError: name 'LLM_MODEL_NAME' is not defined

RAG Helper Functions

In [ ]:
# Question Type Config
QUESTION_TYPE_CONFIG = {
    "directions":    {"top_k": 3, "sem_weight": 0.7, "bm25_weight": 0.3},
    "fare":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "location":      {"top_k": 6, "sem_weight": 0.5, "bm25_weight": 0.5},
    "schedule":      {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "howto":         {"top_k": 5, "sem_weight": 0.8, "bm25_weight": 0.2},
    "accessibility": {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
    "bike":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "general":       {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
}

# Matches your actual source types: scraped_web, gtfs_stop, gtfs_route, gtfs_calendar
QUESTION_TYPE_ALLOWED_SOURCES = {
    "directions":    {"scraped_web", "gtfs_route"},
    "fare":          {"scraped_web", "gtfs_route"},
    "location":      {"scraped_web", "gtfs_stop", "gtfs_route"},
    "schedule":      {"scraped_web", "gtfs_calendar"},
    "howto":         {"scraped_web"},
    "accessibility": {"scraped_web", "gtfs_stop"},
    "bike":          {"scraped_web"},
    "general":       {"scraped_web", "gtfs_route", "gtfs_calendar"},
}

GTFS_HEAVY_PENALTY  = {"gtfs_shape", "gtfs_stop_times"}
GTFS_MEDIUM_PENALTY = {"gtfs_stop", "gtfs_trip"}

SOURCE_TOPIC_BOOSTS = {
    "fare":          ["Fare Information", "Passes", "Reduced Fare", "Where to Buy",
                      "Ventra", "U-Pass", "Student", "How-To: Paying"],
    "accessibility": ["Accessibility", "Disabilities", "ADA"],
    "bike":          ["Bike", "Bicycle", "Bike & Ride"],
    "safety":        ["Safety", "Rules", "Code of Conduct", "Policies"],
    "howto":         ["How-To", "Getting Around", "Travel Information"],
}

# Query Expansion
def expand_query(query: str) -> str:
    expansions = {
        "train":      "train rail L CTA",
        "bus":        "bus route CTA",
        "pass":       "pass fare unlimited rides Ventra 1-day 3-day 7-day 30-day price cost",
        "card":       "Ventra card payment fare transit account",
        "price":      "price cost fare how much dollars",
        "cost":       "cost price fare how much dollars",
        "fare":       "fare price cost how much bus train L ride pay",
        "7-day":      "7-day pass cost price fare $20 unlimited",
        "30-day":     "30-day pass cost price fare $75 unlimited monthly",
        "1-day":      "1-day pass cost price fare $5 unlimited",
        "3-day":      "3-day pass cost price fare $15 unlimited visitor",
        "how much":   "cost price fare dollars pay",
        "ride":       "ride fare cost pay bus train",
        "accessible": "accessible wheelchair disability ADA",
        "bike":       "bike bicycle ride board",
        "free":       "free ride pass reduced fare",
        "student":    "student reduced fare U-Pass school university",
        "senior":     "senior reduced fare elderly",
        "transfer":   "transfer ride connection free within 2 hours",
        "airport":    "O'Hare Midway airport train Blue Orange line",
        "ventra":     "Ventra card account fare payment transit",
    }
    expanded = query
    for term, expansion in expansions.items():
        if term.lower() in query.lower():
            expanded += f" {expansion}"
    return expanded

# Question Type Detection
def detect_question_type(query: str) -> str:
    q = query.lower()
    direction_phrases = [
        "how do i get", "how to get", "how can i get",
        "directions to", "directions from",
        "get from", "travel from", "go from",
        "commute from", "commute to",
        "route from", "route to",
        "way to get", "best way to",
        "take to get to", "which train", "which bus",
        "what train", "what bus", "what line",
        "how do i travel", "how do i commute",
    ]
    if any(phrase in q for phrase in direction_phrases):
        return "directions"

    if any(w in q for w in ["how much", "cost", "price", "fare", "$",
                             "student", "discount", "eligible", "upass",
                             "pass", "transfer", "reduced"]):
        return "fare"
    elif any(w in q for w in ["where", "location", "station", "stop", "address"]):
        return "location"
    elif any(w in q for w in ["when", "time", "hours", "schedule", "open"]):
        return "schedule"
    elif any(w in q for w in ["how do", "how can", "how to", "steps"]):
        return "howto"
    elif any(w in q for w in ["accessible", "wheelchair", "disability"]):
        return "accessibility"
    elif any(w in q for w in ["bike", "bicycle", "electric bike"]):
        return "bike"
    else:
        return "general"

# Source Boost
def get_source_boost(source_title: str, question_type: str) -> float:
    for title_keyword in SOURCE_TOPIC_BOOSTS.get(question_type, []):
        if title_keyword.lower() in source_title.lower():
            return 0.15
    return 0.0

# Hybrid Retrieval
def hybrid_retrieve(query, collection, embed_model, bm25_index, chunks,
                    top_k=5, semantic_weight=0.7, bm25_weight=0.3,
                    question_type="general"):

    allowed_sources = QUESTION_TYPE_ALLOWED_SOURCES.get(question_type, None)

    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    sem_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = 60,
        include          = ["documents", "metadatas", "distances"],
    )

    sem_candidates = {}
    for i in range(len(sem_results["documents"][0])):
        source_type  = sem_results["metadatas"][0][i].get("source_type", "")
        source_title = sem_results["metadatas"][0][i].get("source_title", "")
        score        = 1 - sem_results["distances"][0][i]

        if allowed_sources and source_type not in allowed_sources:
            continue

        if source_type in GTFS_HEAVY_PENALTY:
            score *= 0.2
        elif source_type in GTFS_MEDIUM_PENALTY:
            score *= 1.0 if question_type == "location" else 0.5

        score += get_source_boost(source_title, question_type)

        sem_candidates[i] = {
            "score":        score * semantic_weight,
            "text":         sem_results["documents"][0][i],
            "source_id":    sem_results["metadatas"][0][i].get("source_id", ""),
            "source_title": source_title,
            "source_url":   sem_results["metadatas"][0][i].get("source_url", ""),
            "source_type":  source_type,
        }

    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    max_bm25        = bm25_scores.max() if bm25_scores.max() > 0 else 1
    bm25_norm       = bm25_scores / max_bm25

    for idx, meta in sem_candidates.items():
        for j, chunk in enumerate(chunks):
            if chunk["text"] == meta["text"]:
                meta["score"] += bm25_norm[j] * bm25_weight
                break

    return sorted(sem_candidates.values(), key=lambda x: x["score"], reverse=True)

# Re-ranking
def rerank_chunks(query: str, chunks: list[dict]) -> list[dict]:
    stop_words  = {"the","a","an","is","are","how","what","can","i",
                   "do","does","on","in","of","to","my","me","for"}
    query_terms = set(query.lower().split()) - stop_words
    for chunk in chunks:
        term_hits      = sum(1 for t in query_terms if t in chunk["text"].lower())
        chunk["score"] += term_hits * 0.05
    return sorted(chunks, key=lambda x: x["score"], reverse=True)

# Deduplication
def deduplicate_chunks(chunks: list[dict], max_per_source: int = 2) -> list[dict]:
    seen   = {}
    deduped = []
    for chunk in chunks:
        sid = chunk.get("source_id", "")
        if seen.get(sid, 0) < max_per_source:
            seen[sid] = seen.get(sid, 0) + 1
            deduped.append(chunk)
    return deduped

DIRECTIONS_RESPONSE = """I can't provide turn-by-turn directions, but here are the best tools to plan your CTA trip:

- **CTA Trip Planner**: https://www.transitchicago.com/planatrip/
- **Google Maps**: https://maps.google.com (select Transit mode)
- **Ventra App**: Download the Ventra app for real-time trip planning

These tools will show you exactly which bus or train to take, where to board, and how long your trip will take."""

# Prompt Builder
def build_prompt(query: str, context_chunks: list[dict], question_type: str ="general") -> str:

    if not context_chunks:
        return (
            f"<s>[INST] You are a helpful CTA assistant. "
            f"I don't have information about that. "
            f"Question: {query} [/INST]"
        )

    context   = "\n\n".join([
        f"[Source: {c['source_title']}]\n{c['text']}"
        for c in context_chunks
    ])
    avg_score = sum(c["score"] for c in context_chunks) / len(context_chunks)
    confidence_note = (
        "" if avg_score > 0.3
        else "Note: Answer based on best available context.\n\n"
    )

    return (
        f"<s>[INST] You are a factual Chicago Transit Authority (CTA) assistant. "
        f"Your answers must be based STRICTLY on the provided context. "
        f"\n\nCRITICAL RULES:"
        f"\n- Only state facts explicitly written in the context below."
        f"\n- Do NOT add information from your training data or general knowledge."
        f"\n- Do NOT infer, assume, or extrapolate beyond what the context states."
        f"\n- If a specific detail is not in the context, omit it entirely."
        f"\n- Keep your answer concise and factual."
        f"\n- For any question asking how to get from one place to another, "
        f"direct the user to https://www.transitchicago.com/planatrip/ "
        f"or Google Maps instead of answering directly."
        f"\n- If the context contains nothing relevant, say exactly: "
        f"'I don't have enough information to answer that accurately.'\n\n"
        f"{confidence_note}"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer (using ONLY information from the context above): [/INST]"
    )

# Answer Generation
def generate_answer(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3000, # Max length for input sequences
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens     = 200, # Output token limit
            do_sample          = False,
            temperature        = 0.6,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("✓ All RAG helper functions defined")

✓ All RAG helper functions defined


Main ask() function

In [ ]:
def ask(query: str, show_sources: bool = True) -> str:
    q_type = detect_question_type(query)
    cfg    = QUESTION_TYPE_CONFIG[q_type]
    print(f"Question type: {q_type}")

    expanded_query = expand_query(query)
    if expanded_query != query:
        print(f"🔍 Query expanded")

    candidates = hybrid_retrieve(
        expanded_query, collection, embed_model, bm25_index, chunks,
        top_k           = cfg["top_k"] * 4,
        semantic_weight = cfg["sem_weight"],
        bm25_weight     = cfg["bm25_weight"],
        question_type   = q_type,
    )
    candidates     = rerank_chunks(query, candidates)
    candidates     = deduplicate_chunks(candidates)
    context_chunks = candidates[:cfg["top_k"]]

    prompt = build_prompt(query, context_chunks)
    answer = generate_answer(prompt)

    if show_sources:
        print(f"\n{'─'*60}")
        print(f"Q: {query}")
        print(f"{'─'*60}")
        print(f"A: {answer}")
        print(f"\nSources used:")
        for i, c in enumerate(context_chunks, 1):
            print(f"  {i}. [{c['score']:.0%}] {c['source_title']}  ({c['source_type']})")
            if c["source_url"]:
                print(f"       {c['source_url']}")
        print(f"{'─'*60}\n")

    return answer

print("✓ ask() ready")

✓ ask() ready


Test queries

In [ ]:
ask("How much does a 7-day CTA pass cost?")
ask("Is the CTA wheelchair accessible?")
ask("Can I bring my bike on the train?")
ask("What is the Ventra card and how do I get one?")
ask("I am a graduate student at the University of Chicago, am I eligible for a student discount?")
ask("I live in Rogers Park and want to commute to Hyde Park. What train do I ride?")
ask("Can I bring an electric bike on CTA buses and trains?")
ask("Which CTA stations have bike racks where I can park my bicycle?")

Question type: fare
🔍 Query expanded


NameError: name 'collection' is not defined

Interactive Loop

In [ ]:
print("CTA Chatbot Assistant — Ask me anything!")
print("Type 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not query:
        continue
    ask(query)

Debug Retrieval (If necessary)

In [ ]:
def debug_retrieval(query: str, top_k: int = 10):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    raw_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"],
    )

    print(f"\n{'═'*60}")
    print(f"DEBUG: '{query}'")
    print(f"{'═'*60}")
    print(f"\n── Raw Semantic Results ──")
    for i in range(len(raw_results["documents"][0])):
        score       = 1 - raw_results["distances"][0][i]
        source_type = raw_results["metadatas"][0][i].get("source_type", "")
        title       = raw_results["metadatas"][0][i].get("source_title", "")
        text        = raw_results["documents"][0][i][:150].replace("\n", " ")
        print(f"\n  [{i+1}] Score: {score:.3f} | Type: {source_type}")
        print(f"       Title: {title}")
        print(f"       Text:  {text}...")

    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    top_bm25_idx    = bm25_scores.argsort()[::-1][:top_k]

    print(f"\n── BM25 Keyword Results ──")
    for rank, idx in enumerate(top_bm25_idx, 1):
        chunk = chunks[idx]
        text  = chunk["text"][:150].replace("\n", " ")
        print(f"\n  [{rank}] BM25: {bm25_scores[idx]:.3f} | Type: {chunk.get('source_type','')}")
        print(f"       Title: {chunk.get('source_title','')}")
        print(f"       Text:  {text}...")
    print(f"\n{'═'*60}\n")

# Uncomment to run:
# debug_retrieval("How much does a 7-day CTA pass cost?")

RAGAS Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEYS')

openai_judge = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-3.5-turbo", api_key=OPENAI_API_KEY, temperature=0)
)

local_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
)

print("✓ RAGAS judge ready")

Evaluation Questions

In [ ]:
EVAL_QUESTIONS = [
    {"question": "How much does a 7-day CTA pass cost?",
     "ground_truth": "A 7-day CTA pass costs $20 for unlimited rides."},
    {"question": "What is the full fare for riding the L train?",
     "ground_truth": "The full fare for riding the L train is $2.50."},
    {"question": "What is the full fare for riding a CTA bus?",
     "ground_truth": "The full fare for riding a CTA bus is $2.25."},
    {"question": "How much does a 30-day CTA pass cost?",
     "ground_truth": "A 30-day CTA pass costs $75 for unlimited rides."},
    {"question": "Can I bring my bike on the CTA train?",
     "ground_truth": "Yes, you can bring your bike on CTA trains. During busy periods staff may ask you to wait. Divvy and bikeshare bikes are not allowed."},
    {"question": "Is the CTA wheelchair accessible?",
     "ground_truth": "Yes, all CTA buses are accessible. Many rail stations have accessible features. Check the CTA website for specific station accessibility status."},
    {"question": "What is the Ventra card?",
     "ground_truth": "The Ventra card is a reloadable transit card used to pay fares on CTA buses and trains. It costs $5 but the cost is returned as transit value upon registration."},
    {"question": "Are there reduced fares for seniors?",
     "ground_truth": "Yes, seniors 65 and older are eligible for reduced fares on CTA with an RTA Reduced Fare Permit."},
    {"question": "Can students get a discount on CTA fares?",
     "ground_truth": "Yes, students can get reduced fares. College students may be eligible for a U-Pass. Grade and high school students can get a Student Ventra Card with reduced fare privileges."},
    {"question": "How does a CTA transfer work?",
     "ground_truth": "A CTA transfer allows up to 2 additional rides within 2 hours of the first boarding for free."},
]

print(f"✓ {len(EVAL_QUESTIONS)} evaluation questions ready")

RAG Outputs

In [ ]:
def collect_rag_outputs(eval_questions: list[dict]) -> dict:
    questions, answers, contexts_list, ground_truths = [], [], [], []

    for i, item in enumerate(eval_questions):
        query        = item["question"]
        ground_truth = item["ground_truth"]
        print(f"\n[{i+1}/{len(eval_questions)}] {query}")

        q_type = detect_question_type(query)
        cfg    = QUESTION_TYPE_CONFIG[q_type]
        print(f"  Type: {q_type} | top_k: {cfg['top_k']}")

        expanded_query = expand_query(query)
        candidates     = hybrid_retrieve(
            expanded_query, collection, embed_model, bm25_index, chunks,
            top_k           = cfg["top_k"] * 4,
            semantic_weight = cfg["sem_weight"],
            bm25_weight     = cfg["bm25_weight"],
            question_type   = q_type,
        )
        candidates     = rerank_chunks(query, candidates)
        candidates     = deduplicate_chunks(candidates)
        context_chunks = candidates[:cfg["top_k"]]

        if not context_chunks:
            print(f"  ⚠ No chunks retrieved!")
        else:
            print(f"  ✓ {len(context_chunks)} chunks retrieved")

        prompt = build_prompt(query, context_chunks)
        answer = generate_answer(prompt)
        print(f"  A: {answer[:100]}...")

        questions.append(query)
        answers.append(answer)
        contexts_list.append([c["text"] for c in context_chunks])
        ground_truths.append(ground_truth)

    empty_ctx = sum(1 for c in contexts_list if len(c) == 0)
    print(f"\n{'─'*50}")
    print(f"  Total: {len(questions)} | Empty contexts: {empty_ctx} ← should be 0")
    print(f"{'─'*50}")

    return {
        "question":     questions,
        "answer":       answers,
        "contexts":     contexts_list,
        "ground_truth": ground_truths,
    }

print("Collecting RAG outputs...\n")
rag_outputs = collect_rag_outputs(EVAL_QUESTIONS)

Validate before RAGAS

In [ ]:
print("Validating outputs before RAGAS...\n")
all_valid = True

for i, (q, a, c, g) in enumerate(zip(
    rag_outputs["question"], rag_outputs["answer"],
    rag_outputs["contexts"], rag_outputs["ground_truth"]
)):
    issues = []
    if not a or not a.strip():
        issues.append("empty answer")
    if not c or len(c) == 0:
        issues.append("empty contexts ← causes faithfulness=0")
    if any(p in a.lower() for p in ["i apologize", "i don't have",
                                     "please visit", "contact their"]):
        issues.append("model gave non-answer")
    if issues:
        all_valid = False
        print(f"  ⚠ [{i+1}] {q}")
        print(f"       Issues  : {', '.join(issues)}")
        print(f"       Answer  : {a[:100]}")
        print(f"       Contexts: {len(c)} chunks")
    else:
        print(f"  ✓ [{i+1}] {q[:55]}")

print(f"\n{'✓ All valid — safe to run RAGAS' if all_valid else '✗ Fix issues before running RAGAS'}")

Run RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

def run_ragas_evaluation(rag_outputs):
    eval_dataset = Dataset.from_dict(rag_outputs)
    print(f"✓ Dataset: {len(eval_dataset)} rows\n")
    print("Running RAGAS evaluation...\n")

    results = evaluate(
        dataset          = eval_dataset,
        metrics          = [faithfulness, answer_relevancy,
                            context_precision, context_recall],
        llm              = openai_judge,
        embeddings       = local_embeddings,
        raise_exceptions = False,
    )
    return results.to_pandas(), results

df_results, ragas_scores = run_ragas_evaluation(rag_outputs)

# Aggregate scores
print("\n" + "═"*60)
print("RAGAS EVALUATION RESULTS")
print("═"*60)
for key, label in [("faithfulness","Faithfulness"), ("answer_relevancy","Answer Relevancy"),
                   ("context_precision","Context Precision"), ("context_recall","Context Recall")]:
    try:
        val = df_results[key].dropna().mean()
        print(f"  {label:<20} : {val:.3f}")
    except Exception:
        print(f"  {label:<20} : N/A")
print("═"*60)

Per-Question Scores

In [ ]:
def show_per_question_scores(df):
    question_col = "user_input" if "user_input" in df.columns else "question"
    metric_cols  = ["faithfulness","answer_relevancy","context_precision","context_recall"]

    print("\nPer-Question RAGAS Scores")
    print("─"*105)
    print(f"  {'Question':<52} {'Faith':>7} {'Relev':>7} {'Prec':>7} {'Recall':>7}")
    print("─"*105)

    for _, row in df.iterrows():
        q         = str(row[question_col])
        q_display = q[:49] + "..." if len(q) > 49 else q
        scores    = [f"{row.get(c):.2f}" if pd.notna(row.get(c)) else "  N/A"
                     for c in metric_cols]
        print(f"  {q_display:<52} {scores[0]:>7} {scores[1]:>7} {scores[2]:>7} {scores[3]:>7}")

    print("─"*105)
    print(f"\n⚠ Questions scoring below 0.5:")
    weak_found = False
    for _, row in df.iterrows():
        weak = [f"{c}={row.get(c):.2f}" for c in metric_cols
                if pd.notna(row.get(c)) and row.get(c) < 0.5]
        if weak:
            weak_found = True
            print(f"  • {str(row[question_col])[:70]}")
            print(f"    → {', '.join(weak)}")
    if not weak_found:
        print("  ✓ No questions scored below 0.5")

show_per_question_scores(df_results)

Save and Download All files

In [ ]:
# Save RAGAS results

df_results.to_csv("ragas_results.csv", index=False)
import json
summary = {}
for key in ["faithfulness","answer_relevancy","context_precision","context_recall"]:
    try:
        summary[key] = round(float(df_results[key].dropna().mean()), 4)
    except Exception:
        summary[key] = None

with open("ragas_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save ChromaDB and embeddings
shutil.make_archive("cta_chroma_db", "zip", CHROMA_DB_PATH)
files.download("cta_chroma_db.zip")
files.download("cta_embeddings.pkl")
files.download("ragas_results.csv")
files.download("ragas_summary.json")
print("✓ All files downloaded")


Reload Cell (after runtime restart)

In [ ]:
!pip install chromadb sentence-transformers rank_bm25

import shutil, zipfile, os, pickle, json, warnings
import numpy as np
import torch
from collections import Counter
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import chromadb
import pandas as pd

warnings.filterwarnings("ignore")

EMBEDDINGS_PATH = "cta_embeddings.pkl"
CHROMA_DB_PATH  = "cta_chroma_db"
COLLECTION_NAME = "cta_knowledge_base"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL_NAME  = "mistralai/Mistral-7B-Instruct-v0.2"


def load_embeddings(save_path):
    with open(save_path, "rb") as f:
        data = pickle.load(f)
    print(f"✓ Loaded embeddings (shape: {data['embeddings'].shape})")
    return data["embeddings"], data["chunks"], data["model_name"]

def build_bm25_index(chunks):
    return BM25Okapi([c["text"].lower().split() for c in chunks])

# Unzip ChromaDB
with zipfile.ZipFile("cta_chroma_db.zip", "r") as z:
    z.extractall(CHROMA_DB_PATH)

embeddings, chunks, model_name = load_embeddings(EMBEDDINGS_PATH)
client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = client.get_collection(COLLECTION_NAME)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
bm25_index  = build_bm25_index(chunks)

print(f"✓ ChromaDB: {collection.count()} chunks")
print(f"✓ BM25: {len(chunks)} chunks")
print("✓ Ready — now run Cell 7 (LLM) and Cell 8 (helper functions) before querying")

FileNotFoundError: [Errno 2] No such file or directory: 'cta_chroma_db.zip'

## Testing out if Llama would be a better choice for text generation than Mistrial 7B Instruct (Spoiler warning, it didn't)

## Creating the RAG pipeline from the CTA text data

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y chromadb opentelemetry-api opentelemetry-exporter-otlp-proto-grpc \
    opentelemetry-sdk opentelemetry-proto opentelemetry-exporter-otlp-proto-common \
    opentelemetry-semantic-conventions opentelemetry-exporter-otlp-proto-http opentelemetry-exporter-gcp-logging google-adk -q

!{sys.executable} -m pip uninstall -y langchain langchain-core langchain-community langchain-google-vertexai ragas langchain-openai -q

!{sys.executable} -m pip install -q --upgrade chromadb sentence-transformers rank_bm25 transformers accelerate "bitsandbytes>=0.46.1"
!{sys.executable} -m pip install -q --upgrade langchain langchain-core langchain-community langchain-google-vertexai langchain-openai
!{sys.executable} -m pip install -q --upgrade ragas datasets
print('Install complete')

In [ ]:
# Imports and Configs
import json, os, pickle, shutil, zipfile, warnings
import numpy as np
import torch
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rank_bm25 import BM25Okapi
import chromadb
from google.colab import files, drive

warnings.filterwarnings('ignore')

DATASET_PATH    = 'cta_combined_dataset.json'
EMBEDDINGS_PATH = 'cta_embeddings.pkl'
CHROMA_DB_PATH  = 'cta_chroma_db'
COLLECTION_NAME = 'cta_knowledge_base'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
LLM_MODEL_NAME  = 'NousResearch/Llama-2-7b-chat-hf'

print(f'LLM: {LLM_MODEL_NAME}')

Mounting Drive and Load Datset

In [ ]:
drive.mount('/content/drive')

SOURCE_PATH = '/content/drive/Shareddrives/ML2-Final Project/Data + Embeddings/cta_combined_dataset.json'
if os.path.exists(SOURCE_PATH):
    shutil.copy(SOURCE_PATH, DATASET_PATH)
    print(f'Copied dataset')
else:
    print(f'Not found: {SOURCE_PATH}')

def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    chunks = [c for c in chunks if c.get('text','').strip()]
    print(f'Loaded {len(chunks)} chunks')
    return chunks

chunks = load_dataset(DATASET_PATH)
print('Sample:', json.dumps(chunks[0], indent=2))

Generate or Load Embeddings

In [ ]:
def generate_and_save_embeddings(chunks, model_name, save_path):
    model = SentenceTransformer(model_name)
    texts = [c['text'] for c in chunks]
    embeddings = model.encode(texts, batch_size=128 if torch.cuda.is_available() else 32,
                               show_progress_bar=True, convert_to_numpy=True)
    with open(save_path, 'wb') as f:
        pickle.dump({'embeddings': embeddings, 'chunks': chunks, 'model_name': model_name}, f)
    print(f'Saved embeddings: {embeddings.shape}')
    return embeddings

def load_embeddings(save_path):
    with open(save_path, 'rb') as f:
        data = pickle.load(f)
    print(f'Loaded embeddings: {data["embeddings"].shape}')
    return data['embeddings'], data['chunks'], data['model_name']

def deduplicate_dataset(chunks, embeddings):
    seen, clean_c, clean_e = set(), [], []
    for c, e in zip(chunks, embeddings):
        cid = c.get('chunk_id','')
        if cid not in seen:
            seen.add(cid)
            clean_c.append(c)
            clean_e.append(e)
    print(f'Deduplicated: {len(chunks)} -> {len(clean_c)} chunks')
    return clean_c, np.array(clean_e)

if os.path.exists(EMBEDDINGS_PATH):
    embeddings, chunks, model_name = load_embeddings(EMBEDDINGS_PATH)
else:
    embeddings = generate_and_save_embeddings(chunks, EMBEDDING_MODEL, EMBEDDINGS_PATH)
    model_name = EMBEDDING_MODEL

chunks, embeddings = deduplicate_dataset(chunks, embeddings)
print(f'Ready: {embeddings.shape[0]} chunks x {embeddings.shape[1]} dims')

Build ChromaDB

In [ ]:
def build_chroma_db(chunks, embeddings, db_path, collection_name):
    if os.path.exists(db_path):
        shutil.rmtree(db_path)
    client     = chromadb.PersistentClient(path=db_path)
    collection = client.create_collection(name=collection_name, metadata={'hnsw:space':'cosine'})
    batch_size, total, inserted = 500, len(chunks), 0
    for start in range(0, total, batch_size):
        bc = chunks[start:start+batch_size]
        be = embeddings[start:start+batch_size]
        vc, ve = [], []
        for c, e in zip(bc, be):
            if c.get('chunk_id') and c.get('text','').strip():
                vc.append(c); ve.append(e)
        if not vc: continue
        try:
            collection.add(
                ids        = [c['chunk_id'] for c in vc],
                embeddings = [e.tolist() for e in ve],
                documents  = [c['text'] for c in vc],
                metadatas  = [{'source_id':c.get('source_id',''),'source_title':c.get('source_title',''),
                               'source_url':c.get('source_url',''),'source_type':c.get('source_type','')} for c in vc],
            )
            inserted += len(vc)
        except Exception as e:
            print(f'Batch failed: {e}')
    print(f'ChromaDB ready: {collection.count()} chunks indexed')
    return client, collection

client, collection = build_chroma_db(chunks, embeddings, CHROMA_DB_PATH, COLLECTION_NAME)
all_meta = collection.get(limit=collection.count(), include=['metadatas'])
chroma_types = Counter(m.get('source_type','MISSING') for m in all_meta['metadatas'])
print('Source types:')
for t, n in chroma_types.most_common():
    print(f'  {t:<30} {n}')

Load Embedding Model and Build BM25

In [ ]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f'Embedding model ready: {EMBEDDING_MODEL}')

bm25_index = BM25Okapi([c['text'].lower().split() for c in chunks])
print(f'BM25 index built over {len(chunks)} chunks')

Load LLM

In [ ]:
print(f'Loading {LLM_MODEL_NAME} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = 'auto',
)
print(f'Loaded: {LLM_MODEL_NAME}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

NameError: name 'LLM_MODEL_NAME' is not defined

RAG Helper Functions

Key changes from previous version:
<br> **Problem:**
> The `0.25` score threshold added earlier was silently dropping valid chunks for ~5 of 10 questions,
because Llama-2's embedding space produces lower similarity scores than Mistral for these CTA queries.
Combined with strict per-type source filtering, many questions ended up with 0 retrieved chunks.

 **Fix:**
- Removed the hard `0.25` threshold entirely
- Replaced hard source filtering with a soft penalty system — wrong source types get score penalties instead of being blocked
- `detect_question_type` is now only used to set BM25/semantic weights, not to gate which chunks are allowed
- This guarantees every question gets at least some chunks, which is required for RAGAS to produce non-zero scores

In [ ]:
# Question type config — controls retrieval weights only, not source filtering
QUESTION_TYPE_CONFIG = {
    'fare':          {'top_k': 5, 'sem_weight': 0.4, 'bm25_weight': 0.6},
    'location':      {'top_k': 5, 'sem_weight': 0.5, 'bm25_weight': 0.5},
    'schedule':      {'top_k': 5, 'sem_weight': 0.5, 'bm25_weight': 0.5},
    'howto':         {'top_k': 5, 'sem_weight': 0.6, 'bm25_weight': 0.4},
    'accessibility': {'top_k': 5, 'sem_weight': 0.5, 'bm25_weight': 0.5},
    'bike':          {'top_k': 5, 'sem_weight': 0.4, 'bm25_weight': 0.6},
    'general':       {'top_k': 5, 'sem_weight': 0.5, 'bm25_weight': 0.5},
}

# Soft penalty multipliers by source type
# Instead of hard blocking, irrelevant source types get their scores reduced
# scraped_web always gets full score since it contains the actual CTA content
SOURCE_TYPE_PENALTIES = {
    'scraped_web':    1.0,   # no penalty — this is where all real answers live
    'gtfs_route':     0.6,   # sometimes useful for route/fare questions
    'gtfs_calendar':  0.4,   # rarely useful
    'gtfs_stop':      0.2,   # almost never useful for these question types
    'gtfs_trip':      0.2,
    'gtfs_shape':     0.05,  # nearly useless for Q&A
    'gtfs_stop_times':0.05,
}

# Boost scores when source title strongly matches the question topic
SOURCE_TOPIC_BOOSTS = {
    'fare':          ['Fare Information','Passes','Reduced Fare','Ventra','U-Pass','Student','How-To: Paying'],
    'accessibility': ['Accessibility','Disabilities','ADA'],
    'bike':          ['Bike','Bicycle','Bike & Ride'],
    'howto':         ['How-To','Getting Around','Travel Information'],
    'general':       ['Fare Information','Travel Information','How-To'],
}


def expand_query(query):
    """Adds related keywords to improve BM25 matching."""
    expansions = {
        'train':      'train rail L CTA',
        'bus':        'bus route CTA',
        'pass':       'pass fare unlimited rides Ventra 1-day 3-day 7-day 30-day price cost',
        'card':       'Ventra card payment fare transit account',
        'price':      'price cost fare how much dollars',
        'cost':       'cost price fare how much dollars',
        'fare':       'fare price cost how much bus train L ride pay',
        '7-day':      '7-day pass cost price fare $20 unlimited',
        '30-day':     '30-day pass cost price fare $75 unlimited monthly',
        '1-day':      '1-day pass cost price fare $5 unlimited',
        '3-day':      '3-day pass cost price fare $15 unlimited visitor',
        'how much':   'cost price fare dollars pay',
        'ride':       'ride fare cost pay bus train',
        'accessible': 'accessible wheelchair disability ADA',
        'bike':       'bike bicycle ride board',
        'free':       'free ride pass reduced fare',
        'student':    'student reduced fare U-Pass school university',
        'senior':     'senior reduced fare elderly RTA',
        'seniors':    'senior reduced fare elderly RTA',
        'discount':   'reduced fare discount student senior eligible',
        'transfer':   'transfer ride connection free within 2 hours',
        'airport':    "O'Hare Midway airport train Blue Orange line",
        'ventra':     'Ventra card account fare payment transit reloadable',
        'wheelchair': 'wheelchair accessible disability ADA lift ramp',
    }
    expanded = query
    for term, expansion in expansions.items():
        if term.lower() in query.lower():
            expanded += f' {expansion}'
    return expanded


def detect_question_type(query):
    """Classifies question to set retrieval weights. No longer used for hard source filtering."""
    q = query.lower()
    if any(w in q for w in ['how much','cost','price','fare','$','pass','transfer',
                             'reduced','senior','seniors','student','college',
                             'university','discount','eligible','upass']):
        return 'fare'
    elif any(w in q for w in ['what is ventra','ventra card','how do i get ventra']):
        return 'howto'
    elif any(w in q for w in ['where','location','station','stop','address']):
        return 'location'
    elif any(w in q for w in ['when','time','hours','schedule','open']):
        return 'schedule'
    elif any(w in q for w in ['how do','how can','how to','steps']):
        return 'howto'
    elif any(w in q for w in ['accessible','wheelchair','disability']):
        return 'accessibility'
    elif any(w in q for w in ['bike','bicycle','electric bike']):
        return 'bike'
    else:
        return 'general'


def hybrid_retrieve(query, top_k=5, semantic_weight=0.5, bm25_weight=0.5,
                    question_type='general'):
    """
    Hybrid semantic + BM25 retrieval.
    Uses soft penalties instead of hard source filtering so no question
    is left with 0 chunks due to filtering.
    """
    query_embedding = embed_model.encode([query], convert_to_numpy=True)

    # Retrieve top 80 semantic candidates (more candidates = better BM25 fusion)
    sem_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = 80,
        include          = ['documents', 'metadatas', 'distances'],
    )

    sem_candidates = {}
    for i in range(len(sem_results['documents'][0])):
        source_type  = sem_results['metadatas'][0][i].get('source_type', '')
        source_title = sem_results['metadatas'][0][i].get('source_title', '')
        score        = 1 - sem_results['distances'][0][i]

        # Apply soft penalty by source type instead of hard filtering
        # scraped_web gets 1.0 (no penalty), gtfs_stop gets 0.2, etc.
        penalty = SOURCE_TYPE_PENALTIES.get(source_type, 0.3)
        score  *= penalty

        # Boost score if source title matches the question topic
        for kw in SOURCE_TOPIC_BOOSTS.get(question_type, []):
            if kw.lower() in source_title.lower():
                score += 0.15
                break

        sem_candidates[i] = {
            'score':        score * semantic_weight,
            'text':         sem_results['documents'][0][i],
            'source_id':    sem_results['metadatas'][0][i].get('source_id', ''),
            'source_title': source_title,
            'source_url':   sem_results['metadatas'][0][i].get('source_url', ''),
            'source_type':  source_type,
        }

    # BM25 scoring over all chunks
    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    max_bm25        = bm25_scores.max() if bm25_scores.max() > 0 else 1
    bm25_norm       = bm25_scores / max_bm25

    # Add BM25 score to semantic candidates
    for idx, meta in sem_candidates.items():
        for j, chunk in enumerate(chunks):
            if chunk['text'] == meta['text']:
                meta['score'] += bm25_norm[j] * bm25_weight
                break

    return sorted(sem_candidates.values(), key=lambda x: x['score'], reverse=True)


def rerank_chunks(query, chunk_list):
    """Boosts chunks that contain exact query terms."""
    stop_words  = {'the','a','an','is','are','how','what','can','i',
                   'do','does','on','in','of','to','my','me','for','there'}
    query_terms = set(query.lower().split()) - stop_words
    for chunk in chunk_list:
        term_hits      = sum(1 for t in query_terms if t in chunk['text'].lower())
        chunk['score'] += term_hits * 0.05
    return sorted(chunk_list, key=lambda x: x['score'], reverse=True)


def deduplicate_chunks(chunk_list, max_per_source=2):
    """Limits chunks per source to avoid one page dominating context."""
    seen, deduped = {}, []
    for chunk in chunk_list:
        sid = chunk.get('source_id', '')
        if seen.get(sid, 0) < max_per_source:
            seen[sid] = seen.get(sid, 0) + 1
            deduped.append(chunk)
    return deduped


def build_prompt(query, context_chunks):
    """
    Builds a Llama-2 chat formatted prompt.
    System message goes inside <<SYS>> tags.
    Context and question go in the user turn after <<SYS>>.
    """
    system_msg = (
        'You are a helpful Chicago Transit Authority (CTA) assistant. '
        'Answer the question using ONLY the context provided. '
        'Be concise and specific. Include prices, times, or details when available. '
        'Only say you do not have information if the context contains absolutely '
        'nothing relevant to the question.'
    )
    if not context_chunks:
        return ('<s>[INST] <<SYS>>\n' + system_msg + '\n<</SYS>>\n\n'
                'Question: ' + query + ' [/INST]')
    context = '\n\n'.join(
        f"[Source: {c['source_title']}]\n{c['text']}" for c in context_chunks
    )
    return (
        '<s>[INST] <<SYS>>\n' + system_msg + '\n<</SYS>>\n\n'
        'Context:\n' + context + '\n\nQuestion: ' + query + ' [/INST]'
    )


def generate_answer(prompt):
    """Runs the prompt through Llama-2 and returns the generated text."""
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=3000
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 200,
            do_sample          = False,
            temperature        = 1.0,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print('All RAG helper functions ready')

Main ask() function

In [ ]:
def ask(query, show_sources=True):
    q_type = detect_question_type(query)
    cfg    = QUESTION_TYPE_CONFIG[q_type]

    expanded_query = expand_query(query)
    candidates     = hybrid_retrieve(
        expanded_query,
        top_k           = cfg['top_k'] * 4,
        semantic_weight = cfg['sem_weight'],
        bm25_weight     = cfg['bm25_weight'],
        question_type   = q_type,
    )
    candidates     = rerank_chunks(query, candidates)
    candidates     = deduplicate_chunks(candidates)
    context_chunks = candidates[:cfg['top_k']]

    prompt = build_prompt(query, context_chunks)
    answer = generate_answer(prompt)

    if show_sources:
        print(f'\n{"─"*60}')
        print(f'Q [{q_type}]: {query}')
        print(f'{"─"*60}')
        print(f'A: {answer}')
        print(f'\nSources ({len(context_chunks)} chunks):')
        for i, c in enumerate(context_chunks, 1):
            print(f'  {i}. [{c["score"]:.2f}] {c["source_title"]} ({c["source_type"]})')
        print(f'{"─"*60}\n')
    return answer

print('ask() ready')

Test queries

In [ ]:
ask('How much does a 7-day CTA pass cost?')
ask('Is the CTA wheelchair accessible?')
ask('Can I bring my bike on the train?')
ask('What is the Ventra card and how do I get one?')
ask('I am a graduate student at the University of Chicago, am I eligible for a student discount?')
ask('I live in Rogers Park and want to commute to Hyde Park. What train do I ride?')
ask('Can I bring an electric bike on CTA buses and trains?')
ask('Which CTA stations have bike racks where I can park my bicycle?')

#ask('How much does a 7-day CTA pass cost?')
#ask('Is the CTA wheelchair accessible?')
#ask('Can I bring my bike on the train?')
#ask('What is the Ventra card and how do I get one?')
ask('Are there reduced fares for seniors?')
ask('Can students get a discount on CTA fares?')
ask('How does a CTA transfer work?')


Interactive loop

In [ ]:
print('CTA Chatbot (Llama-2) — Type your question or "quit" to exit.\n')
while True:
    query = input('You: ').strip()
    if query.lower() in ('quit', 'exit', 'q'):
        print('Goodbye!')
        break
    if not query:
        continue
    ask(query)

Debug Retrieval (If necessary)

In [ ]:
def debug_retrieval(query, top_k=10):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    raw = collection.query(query_embeddings=query_embedding.tolist(), n_results=top_k,
                           include=['documents','metadatas','distances'])
    print(f'\nDEBUG: "{query}"')
    for i in range(len(raw['documents'][0])):
        score = 1 - raw['distances'][0][i]
        stype = raw['metadatas'][0][i].get('source_type','')
        title = raw['metadatas'][0][i].get('source_title','')
        text  = raw['documents'][0][i][:100].replace('\n',' ')
        print(f'  [{i+1}] {score:.3f} | {stype} | {title}')
        print(f'       {text}...')

# Uncomment to run:
# debug_retrieval('How much does a 7-day CTA pass cost?')
# debug_retrieval('Are there reduced fares for seniors?')
# debug_retrieval('Can students get a discount on CTA fares?')

RAGAS Setup

In [ ]:
import sys
from types import ModuleType

# Fix ragas internal dependency on ChatVertexAI which moved packages
class _VertexShim(ModuleType):
    def __getattr__(self, name):
        if name == 'ChatVertexAI':
            from langchain_google_vertexai import ChatVertexAI
            return ChatVertexAI
        raise AttributeError(name)

sys.modules['langchain_community.chat_models.vertexai'] = _VertexShim('langchain_community.chat_models.vertexai')

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEYS')

openai_judge = LangchainLLMWrapper(
    ChatOpenAI(model='gpt-3.5-turbo', api_key=OPENAI_API_KEY, temperature=0)
)
local_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
)
print('RAGAS judge ready')

Evaluation Questions

In [ ]:
EVAL_QUESTIONS = [
    {'question': 'How much does a 7-day CTA pass cost?',
     'ground_truth': 'A 7-day CTA pass costs $20 for unlimited rides.'},
    {'question': 'What is the full fare for riding the L train?',
     'ground_truth': 'The full fare for riding the L train is $2.50.'},
    {'question': 'What is the full fare for riding a CTA bus?',
     'ground_truth': 'The full fare for riding a CTA bus is $2.25.'},
    {'question': 'How much does a 30-day CTA pass cost?',
     'ground_truth': 'A 30-day CTA pass costs $75 for unlimited rides.'},
    {'question': 'Can I bring my bike on the CTA train?',
     'ground_truth': 'Yes, you can bring your bike on CTA trains. During busy periods staff may ask you to wait. Divvy and bikeshare bikes are not allowed.'},
    {'question': 'Is the CTA wheelchair accessible?',
     'ground_truth': 'Yes, all CTA buses are accessible. Many rail stations have accessible features. Check the CTA website for specific station accessibility status.'},
    {'question': 'What is the Ventra card?',
     'ground_truth': 'The Ventra card is a reloadable transit card used to pay fares on CTA buses and trains. It costs $5 but the cost is returned as transit value upon registration.'},
    {'question': 'Are there reduced fares for seniors?',
     'ground_truth': 'Yes, seniors 65 and older are eligible for reduced fares on CTA with an RTA Reduced Fare Permit.'},
    {'question': 'Can students get a discount on CTA fares?',
     'ground_truth': 'Yes, students can get reduced fares. College students may be eligible for a U-Pass. Grade and high school students can get a Student Ventra Card with reduced fare privileges.'},
    {'question': 'How does a CTA transfer work?',
     'ground_truth': 'A CTA transfer allows up to 2 additional rides within 2 hours of the first boarding for free.'},
]
print(f'{len(EVAL_QUESTIONS)} evaluation questions ready')



RAG Outputs

In [ ]:
# Column names match RAGAS >= 0.2 requirements: `user_input`, `response`, `retrieved_contexts`, `reference`
def collect_rag_outputs(eval_questions):
    user_inputs, answers, contexts_list, ground_truths = [], [], [], []

    for i, item in enumerate(eval_questions):
        query        = item['question']
        ground_truth = item['ground_truth']
        print(f'\n[{i+1}/{len(eval_questions)}] {query}')

        q_type = detect_question_type(query)
        cfg    = QUESTION_TYPE_CONFIG[q_type]

        expanded_query = expand_query(query)
        candidates     = hybrid_retrieve(
            expanded_query,
            top_k           = cfg['top_k'] * 4,
            semantic_weight = cfg['sem_weight'],
            bm25_weight     = cfg['bm25_weight'],
            question_type   = q_type,
        )
        candidates     = rerank_chunks(query, candidates)
        candidates     = deduplicate_chunks(candidates)
        context_chunks = candidates[:cfg['top_k']]

        print(f'  {len(context_chunks)} chunks retrieved | type: {q_type}')
        if context_chunks:
            print(f'  Top source: {context_chunks[0]["source_title"]} ({context_chunks[0]["source_type"]})')

        prompt   = build_prompt(query, context_chunks)
        answer   = generate_answer(prompt)
        contexts = [c['text'] for c in context_chunks]

        print(f'  A: {answer[:100]}...')

        user_inputs.append(query)
        answers.append(answer)
        contexts_list.append(contexts)
        ground_truths.append(ground_truth)

    empty_ctx = sum(1 for c in contexts_list if len(c) == 0)
    print(f'\nDone: {len(user_inputs)} questions | {empty_ctx} with empty context')

    return {
        'user_input':         user_inputs,
        'response':           answers,
        'retrieved_contexts': contexts_list,
        'reference':          ground_truths,
    }

print('Collecting RAG outputs...\n')
rag_outputs = collect_rag_outputs(EVAL_QUESTIONS)

Validate before RAGAS
> Note: the non-answer check is intentionally lenient — it only flags truly empty answers.
> Partial answers like 'I do not have complete information but...' are acceptable for RAGAS scoring.


In [ ]:
print('Validating outputs...\n')
all_valid = True

for i, (q, a, c, g) in enumerate(zip(
    rag_outputs['user_input'], rag_outputs['response'],
    rag_outputs['retrieved_contexts'], rag_outputs['reference']
)):
    issues = []
    if not a or not a.strip():
        issues.append('empty answer')
    if not c or len(c) == 0:
        # Empty context is flagged but we still proceed — RAGAS will score 0
        # for context metrics but answer_relevancy can still score
        issues.append('empty contexts')
    if issues:
        all_valid = False
        print(f'  WARNING [{i+1}] {q}')
        print(f'    Issues  : {", ".join(issues)}')
        print(f'    Answer  : {a[:120]}')
        print(f'    Contexts: {len(c)} chunks')
    else:
        print(f'  OK [{i+1}] {q[:60]} ({len(c)} chunks)')

status = 'All valid' if all_valid else 'Some warnings — see above'
print(f'\n{status}')
print('Proceeding to RAGAS regardless — partial results are still useful for comparison.')

Run RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

def run_ragas_evaluation(rag_outputs):
    eval_dataset = Dataset.from_dict(rag_outputs)
    print(f'Dataset: {len(eval_dataset)} rows')
    print(f'Columns: {eval_dataset.column_names}\n')
    results = evaluate(
        dataset          = eval_dataset,
        metrics          = [faithfulness, answer_relevancy, context_precision, context_recall],
        llm              = openai_judge,
        embeddings       = local_embeddings,
        raise_exceptions = False,
    )
    return results.to_pandas(), results

df_results, ragas_scores = run_ragas_evaluation(rag_outputs)

print('\n' + '='*60)
print('RAGAS RESULTS — Llama-2')
print('='*60)
for key, label in [('faithfulness','Faithfulness'),('answer_relevancy','Answer Relevancy'),
                   ('context_precision','Context Precision'),('context_recall','Context Recall')]:
    try:
        val = df_results[key].dropna().mean()
        print(f'  {label:<22} : {val:.3f}')
    except Exception:
        print(f'  {label:<22} : N/A')
print('='*60)

Per-Question Scores

In [ ]:
def show_per_question_scores(df):
    question_col = 'user_input' if 'user_input' in df.columns else 'question'
    metric_cols  = ['faithfulness','answer_relevancy','context_precision','context_recall']
    print('\nPer-Question RAGAS Scores — Llama-2')
    print('-'*105)
    print(f'  {"Question":<52} {"Faith":>7} {"Relev":>7} {"Prec":>7} {"Recall":>7}')
    print('-'*105)
    for _, row in df.iterrows():
        q         = str(row[question_col])
        q_display = q[:49] + '...' if len(q) > 49 else q
        scores    = [f'{row.get(c):.2f}' if pd.notna(row.get(c)) else '  N/A' for c in metric_cols]
        print(f'  {q_display:<52} {scores[0]:>7} {scores[1]:>7} {scores[2]:>7} {scores[3]:>7}')
    print('-'*105)

show_per_question_scores(df_results)

Save and download all files

In [ ]:
df_results.to_csv('ragas_results_llama2.csv', index=False)

summary = {}
for key in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    try:
        summary[key] = round(float(df_results[key].dropna().mean()), 4)
    except Exception:
        summary[key] = None

with open('ragas_summary_llama2.json', 'w') as f:
    json.dump(summary, f, indent=2)

shutil.make_archive('cta_chroma_db', 'zip', CHROMA_DB_PATH)

files.download('ragas_results_llama2.csv')
files.download('ragas_summary_llama2.json')
files.download('cta_chroma_db.zip')
files.download('cta_embeddings.pkl')
print('All files downloaded')

Reload Cell (after runtime restart)

In [ ]:
# Uncomment and run this entire cell after a Colab runtime restart.
# Then skip directly to Cell 7 (Load LLM) and Cell 8 (RAG helpers).

# import sys
# !{sys.executable} -m pip install -q chromadb sentence-transformers rank_bm25
#
# import shutil, zipfile, os, pickle, json, warnings
# import numpy as np, torch, pandas as pd
# from collections import Counter
# from sentence_transformers import SentenceTransformer
# from rank_bm25 import BM25Okapi
# import chromadb
# from google.colab import files, drive
# warnings.filterwarnings('ignore')
#
# EMBEDDINGS_PATH = 'cta_embeddings.pkl'
# CHROMA_DB_PATH  = 'cta_chroma_db'
# COLLECTION_NAME = 'cta_knowledge_base'
# EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
# LLM_MODEL_NAME  = 'NousResearch/Llama-2-7b-chat-hf'
#
# drive.mount('/content/drive')
# with zipfile.ZipFile('cta_chroma_db.zip', 'r') as z:
#     z.extractall(CHROMA_DB_PATH)
# with open(EMBEDDINGS_PATH, 'rb') as f:
#     data = pickle.load(f)
# embeddings  = data['embeddings']
# chunks      = data['chunks']
# client      = chromadb.PersistentClient(path=CHROMA_DB_PATH)
# collection  = client.get_collection(COLLECTION_NAME)
# embed_model = SentenceTransformer(EMBEDDING_MODEL)
# bm25_index  = BM25Okapi([c['text'].lower().split() for c in chunks])
# print(f'ChromaDB: {collection.count()} chunks | BM25: {len(chunks)} chunks')
# print('Ready — now run Cell 7 then Cell 8')


## Our interface to interact with the CTA chatbot via Gradio

# CTA RAG Chatbot — Gradio App Demo
**App demo component for ML2 Final Project**

### Prerequisites
1. Run `RAG_LLM.ipynb` at least once and confirm `cta_embeddings.pkl` and `cta_chroma_db.zip` are saved to the shared Drive folder.
2. Update `DRIVE_BASE` in Cell 2 to match your shared Drive path.

Install dependencies

In [ ]:
# Uninstall packages that conflict with ChromaDB on Colab
# Aggressively uninstall all opentelemetry packages, then chromadb
!pip uninstall -y \
    chromadb \
    opentelemetry-api \
    opentelemetry-sdk \
    opentelemetry-proto \
    opentelemetry-exporter-otlp-proto-grpc \
    opentelemetry-exporter-otlp-proto-http \
    opentelemetry-exporter-otlp-proto-common \
    opentelemetry-semantic-conventions \
    opentelemetry-instrumentation-botocore \
    opentelemetry-instrumentation-fastapi \
    opentelemetry-instrumentation-grpc \
    opentelemetry-instrumentation-httpx \
    opentelemetry-instrumentation-requests \
    opentelemetry-instrumentation-urllib3 \
    opentelemetry-instrumentation-wsgi \
    -q

# Now install chromadb with force-reinstall, allowing it to pull compatible opentelemetry
!pip install -q chromadb --upgrade --force-reinstall

# Install other core dependencies
!pip install -q sentence-transformers rank_bm25
!pip install -q transformers accelerate "bitsandbytes>=0.46.1"
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3

## Configuration

In [ ]:
import os

# ── Drive path ----------------------------------------------------------------
# Update to match your shared Drive folder
DRIVE_BASE        = "/content/drive/Shareddrives/ML2-Final Project/Data + Embeddings"
EMBEDDINGS_PATH   = os.path.join(DRIVE_BASE, "cta_embeddings.pkl")
CHROMA_ZIP_PATH   = os.path.join(DRIVE_BASE, "cta_chroma_db.zip")   # preferred
CHROMA_DIR_PATH   = os.path.join(DRIVE_BASE, "cta_chroma_db")       # fallback
LOCAL_CHROMA_PATH = "/content/cta_chroma_db"  # unzipped here at runtime for speed
COLLECTION_NAME   = "cta_knowledge_base"
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"

# ── LLM -----------------------------------------------------------------------
# Use Jacob's Mistral model for the demo (best-performing).
# To switch to Llama-2, change to: "NousResearch/Llama-2-7b-chat-hf" and set PROMPT_FORMAT = "llama2"
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
PROMPT_FORMAT  = "mistral"   # "mistral" or "llama2"

print("Config loaded")
print(f"  LLM  : {LLM_MODEL_NAME}")
print(f"  Drive: {DRIVE_BASE}")

Config loaded
  LLM  : mistralai/Mistral-7B-Instruct-v0.2
  Drive: /content/drive/Shareddrives/ML2-Final Project/Data + Embeddings


Mount Drive and Load Data

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y numpy scipy sentence-transformers chromadb
!{sys.executable} -m pip install numpy==1.25.2 scipy==1.11.4 sentence-transformers==2.2.2 chromadb==1.5.9

import pickle
import shutil
import zipfile
import numpy as np
import chromadb
import warnings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google.colab import drive

warnings.filterwarnings("ignore")
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Error mounting drive, attempting to unmount and remount: {e}")
    try:
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"Failed to unmount and remount: {e}")
        raise # Re-raise the exception if remounting also fails

# Load embeddings from Drive
print("Loading embeddings...")
with open(EMBEDDINGS_PATH, "rb") as f:
    data = pickle.load(f)
embeddings = data["embeddings"]
chunks     = data["chunks"]
print(f"Loaded {len(chunks)} chunks  (shape: {embeddings.shape})")

# Unzip ChromaDB to local /content for faster reads than Drive
if os.path.exists(LOCAL_CHROMA_PATH):
    shutil.rmtree(LOCAL_CHROMA_PATH)

if os.path.exists(CHROMA_ZIP_PATH):
    print("Unzipping ChromaDB...")
    with zipfile.ZipFile(CHROMA_ZIP_PATH, "r") as z:
        z.extractall(LOCAL_CHROMA_PATH)
    print(f"ChromaDB unzipped to {LOCAL_CHROMA_PATH}")
elif os.path.exists(CHROMA_DIR_PATH):
    print("Copying ChromaDB folder...")
    shutil.copytree(CHROMA_DIR_PATH, LOCAL_CHROMA_PATH)
else:
    raise FileNotFoundError(
        f"ChromaDB not found at:\n  {CHROMA_ZIP_PATH}\n  {CHROMA_DIR_PATH}\n"
        "Run RAG_LLM.ipynb first and save outputs to Drive."
    )

chroma_client = chromadb.PersistentClient(path=LOCAL_CHROMA_PATH)
collection    = chroma_client.get_collection(COLLECTION_NAME)
print(f"ChromaDB connected: {collection.count()} chunks indexed")

embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding model ready: {EMBEDDING_MODEL}")

bm25_index = BM25Okapi([c["text"].lower().split() for c in chunks])
print(f"BM25 index built over {len(chunks)} chunks")

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.11.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached scipy-1.11.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.8 MB)
Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.6 MB/s eta 0:00:00
   ━━━━

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

Load LLM

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Loading {LLM_MODEL_NAME} in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

print(f"Loaded: {LLM_MODEL_NAME}")
if torch.cuda.is_available():
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB used")

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

Full RAG Pipeline

In [ ]:
# ── Question Type Config -----------------------------------------------------
QUESTION_TYPE_CONFIG = {
    "fare":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "location":      {"top_k": 6, "sem_weight": 0.5, "bm25_weight": 0.5},
    "schedule":      {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "howto":         {"top_k": 5, "sem_weight": 0.8, "bm25_weight": 0.2},
    "accessibility": {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
    "bike":          {"top_k": 5, "sem_weight": 0.6, "bm25_weight": 0.4},
    "general":       {"top_k": 5, "sem_weight": 0.7, "bm25_weight": 0.3},
}

QUESTION_TYPE_ALLOWED_SOURCES = {
    "fare":          {"scraped_web", "gtfs_route"},
    "location":      {"scraped_web", "gtfs_stop", "gtfs_route"},
    "schedule":      {"scraped_web", "gtfs_calendar"},
    "howto":         {"scraped_web"},
    "accessibility": {"scraped_web", "gtfs_stop"},
    "bike":          {"scraped_web"},
    "general":       {"scraped_web", "gtfs_route", "gtfs_calendar"},
}

GTFS_HEAVY_PENALTY  = {"gtfs_shape", "gtfs_stop_times"}
GTFS_MEDIUM_PENALTY = {"gtfs_stop", "gtfs_trip"}

SOURCE_TOPIC_BOOSTS = {
    "fare":          ["Fare Information", "Passes", "Reduced Fare", "Where to Buy",
                      "Ventra", "U-Pass", "Student", "How-To: Paying"],
    "accessibility": ["Accessibility", "Disabilities", "ADA"],
    "bike":          ["Bike", "Bicycle", "Bike & Ride"],
    "safety":        ["Safety", "Rules", "Code of Conduct", "Policies"],
    "howto":         ["How-To", "Getting Around", "Travel Information"],
}


def expand_query(query):
    expansions = {
        "train":      "train rail L CTA",
        "bus":        "bus route CTA",
        "pass":       "pass fare unlimited rides Ventra 1-day 3-day 7-day 30-day price cost",
        "card":       "Ventra card payment fare transit account",
        "price":      "price cost fare how much dollars",
        "cost":       "cost price fare how much dollars",
        "fare":       "fare price cost how much bus train L ride pay",
        "7-day":      "7-day pass cost price fare $20 unlimited",
        "30-day":     "30-day pass cost price fare $75 unlimited monthly",
        "1-day":      "1-day pass cost price fare $5 unlimited",
        "3-day":      "3-day pass cost price fare $15 unlimited visitor",
        "how much":   "cost price fare dollars pay",
        "ride":       "ride fare cost pay bus train",
        "accessible": "accessible wheelchair disability ADA",
        "bike":       "bike bicycle ride board",
        "free":       "free ride pass reduced fare",
        "student":    "student reduced fare U-Pass school university",
        "senior":     "senior reduced fare elderly",
        "transfer":   "transfer ride connection free within 2 hours",
        "airport":    "O'Hare Midway airport train Blue Orange line",
        "ventra":     "Ventra card account fare payment transit",
    }
    expanded = query
    for term, expansion in expansions.items():
        if term.lower() in query.lower():
            expanded += f" {expansion}"
    return expanded


def detect_question_type(query):
    q = query.lower()
    if any(w in q for w in ["how much", "cost", "price", "fare", "$",
                             "student", "discount", "eligible", "upass",
                             "pass", "transfer", "reduced"]):
        return "fare"
    elif any(w in q for w in ["where", "location", "station", "stop", "address"]):
        return "location"
    elif any(w in q for w in ["when", "time", "hours", "schedule", "open"]):
        return "schedule"
    elif any(w in q for w in ["how do", "how can", "how to", "steps"]):
        return "howto"
    elif any(w in q for w in ["accessible", "wheelchair", "disability"]):
        return "accessibility"
    elif any(w in q for w in ["bike", "bicycle", "electric bike"]):
        return "bike"
    else:
        return "general"


def get_source_boost(source_title, question_type):
    for kw in SOURCE_TOPIC_BOOSTS.get(question_type, []):
        if kw.lower() in source_title.lower():
            return 0.15
    return 0.0


def hybrid_retrieve(query, top_k=5, semantic_weight=0.7, bm25_weight=0.3,
                    question_type="general"):
    allowed_sources = QUESTION_TYPE_ALLOWED_SOURCES.get(question_type, None)
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    sem_results = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results        = 60,
        include          = ["documents", "metadatas", "distances"],
    )
    sem_candidates = {}
    for i in range(len(sem_results["documents"][0])):
        source_type  = sem_results["metadatas"][0][i].get("source_type", "")
        source_title = sem_results["metadatas"][0][i].get("source_title", "")
        score        = 1 - sem_results["distances"][0][i]
        if allowed_sources and source_type not in allowed_sources:
            continue
        if source_type in GTFS_HEAVY_PENALTY:
            score *= 0.2
        elif source_type in GTFS_MEDIUM_PENALTY:
            score *= 1.0 if question_type == "location" else 0.5
        score += get_source_boost(source_title, question_type)
        sem_candidates[i] = {
            "score":        score * semantic_weight,
            "text":         sem_results["documents"][0][i],
            "source_id":    sem_results["metadatas"][0][i].get("source_id", ""),
            "source_title": source_title,
            "source_url":   sem_results["metadatas"][0][i].get("source_url", ""),
            "source_type":  source_type,
        }
    tokenized_query = query.lower().split()
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    max_bm25        = bm25_scores.max() if bm25_scores.max() > 0 else 1
    bm25_norm       = bm25_scores / max_bm25
    for idx, meta in sem_candidates.items():
        for j, chunk in enumerate(chunks):
            if chunk["text"] == meta["text"]:
                meta["score"] += bm25_norm[j] * bm25_weight
                break
    return sorted(sem_candidates.values(), key=lambda x: x["score"], reverse=True)


def rerank_chunks(query, chunk_list):
    stop_words  = {"the","a","an","is","are","how","what","can","i",
                   "do","does","on","in","of","to","my","me","for"}
    query_terms = set(query.lower().split()) - stop_words
    for chunk in chunk_list:
        term_hits      = sum(1 for t in query_terms if t in chunk["text"].lower())
        chunk["score"] += term_hits * 0.05
    return sorted(chunk_list, key=lambda x: x["score"], reverse=True)


def deduplicate_chunks(chunk_list, max_per_source=2):
    seen, deduped = {}, []
    for chunk in chunk_list:
        sid = chunk.get("source_id", "")
        if seen.get(sid, 0) < max_per_source:
            seen[sid] = seen.get(sid, 0) + 1
            deduped.append(chunk)
    return deduped


def build_prompt(query, context_chunks):
    """Format-aware prompt: supports both Mistral and Llama-2 chat templates."""
    system_msg = (
        "You are a helpful Chicago Transit Authority (CTA) assistant. "
        "Answer the question using ONLY the context provided. "
        "Be concise and specific. Include prices, times, or details when available. "
        "If the user asks how to get from point A to point B, recommend Google Maps "
        "or the CTA trip planner at transitchicago.com. "
        "Only say you don't have information if the context contains absolutely "
        "nothing relevant to the question."
    )
    if not context_chunks:
        if PROMPT_FORMAT == "llama2":
            return (
                "<s>[INST] <<SYS>>\n" + system_msg + "\n<</SYS>>\n\n"
                "I don't have enough information to answer that accurately. [/INST]"
            )
        else:
            return (
                "<s>[INST] " + system_msg +
                " I don't have enough information to answer that accurately. [/INST]"
            )
    context   = "\n\n".join(
        f"[Source: {c['source_title']}]\n{c['text']}" for c in context_chunks
    )
    avg_score = sum(c["score"] for c in context_chunks) / len(context_chunks)
    note      = "" if avg_score > 0.3 else "Note: Answer based on best available context.\n\n"
    if PROMPT_FORMAT == "llama2":
        return (
            "<s>[INST] <<SYS>>\n" + system_msg + "\n<</SYS>>\n\n"
            + note + "Context:\n" + context + "\n\nQuestion: " + query + " [/INST]"
        )
    else:
        return (
            "<s>[INST] " + system_msg + "\n\n"
            + note + "Context:\n" + context + "\n\nQuestion: " + query + " [/INST]"
        )


def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3000,
    ).to(llm_model.device)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens     = 200,
            do_sample          = False,
            temperature        = 1.0,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def ask(query):
    """Full RAG pipeline. Returns (answer_str, sources_markdown_str)."""
    q_type = detect_question_type(query)
    cfg    = QUESTION_TYPE_CONFIG[q_type]
    expanded_query = expand_query(query)
    candidates     = hybrid_retrieve(
        expanded_query,
        top_k           = cfg["top_k"] * 4,
        semantic_weight = cfg["sem_weight"],
        bm25_weight     = cfg["bm25_weight"],
        question_type   = q_type,
    )
    candidates     = rerank_chunks(query, candidates)
    candidates     = deduplicate_chunks(candidates)
    context_chunks = candidates[:cfg["top_k"]]
    prompt         = build_prompt(query, context_chunks)
    answer         = generate_answer(prompt)
    seen_urls, source_lines = set(), []
    for c in context_chunks:
        url   = c.get("source_url", "")
        title = c.get("source_title", "Unknown source")
        key   = url if url else title
        if key not in seen_urls:
            seen_urls.add(key)
            source_lines.append(f"- [{title}]({url})" if url else f"- {title}")
    sources_md = "\n".join(source_lines) if source_lines else "_No sources retrieved_"
    return answer, sources_md


print("All RAG helper functions ready")


Launch Gradio App

In [ ]:
import gradio as gr

CTA_RED        = "#C8102E"
CTA_BLUE       = "#003A8F"
CTA_LIGHT_GRAY = "#F5F5F5"
CTA_DARK_GRAY  = "#565A5C"

SUGGESTED_QUESTIONS = [
    "How much does a 7-day CTA pass cost?",
    "What is the Ventra card and how do I get one?",
    "Is the CTA wheelchair accessible?",
    "Can I bring my bike on the train?",
    "Are there student discounts on CTA fares?",
    "How do CTA transfers work?",
]

def chat(user_input, history):
    """One chat turn. Returns (updated_chatbot, updated_state, cleared_textbox)."""
    if not user_input.strip():
        return history, history, ""
    answer, sources_md = ask(user_input)
    full_response = answer
    if sources_md and sources_md != "_No sources retrieved_":
        full_response += "\n\n---\n**Sources:**\n" + sources_md
    history.append((user_input, full_response))
    return history, history, ""


def use_suggestion(suggestion):
    """Populates the text box when a suggestion chip is clicked."""
    return suggestion

custom_css = """
.gradio-container { background-color: #F5F5F5 !important; }
#header-bar {
    background: linear-gradient(90deg, #003A8F 0%, #C8102E 100%);
    padding: 1.2rem 1.5rem;
    border-radius: 10px;
    margin-bottom: 0.5rem;
}
#header-bar h1 { color: white !important; font-size: 1.7rem; margin: 0; }
#header-bar p  { color: rgba(255,255,255,0.85) !important; margin: 0.2rem 0 0; font-size: 0.95rem; }
".message.user { background-color: #003A8F !important; color: white !important; }\n",
".message.bot  { background-color: white !important; border-left: 3px solid #C8102E; }\n",
"/* Gradio 4.x uses these classes instead of .message.bot */\n",
".bubble-wrap .user { background-color: #003A8F !important; color: white !important; }\n",
".bubble-wrap .bot  { background-color: #EEF2FA !important; color: #1a1a1a !important; border-left: 3px solid #C8102E; }\n",
"div[data-testid='bot'] { background-color: #EEF2FA !important; color: #1a1a1a !important; }\n",
"div[data-testid='user'] { background-color: #003A8F !important; color: white !important; }\n",
#send-btn { background-color: #C8102E !important; color: white !important; border-radius: 8px; }
#send-btn:hover { background-color: #a00d24 !important; }
.suggestion-btn {
    background-color: white !important;
    border: 1px solid #003A8F !important;
    color: #003A8F !important;
    border-radius: 20px !important;
    font-size: 0.82rem !important;
    padding: 4px 12px !important;
}
.suggestion-btn:hover { background-color: #003A8F !important; color: white !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="CTA Travel Assistant") as demo:

    # Header
    with gr.Column(elem_id="header-bar"):
        gr.HTML(
            "<h1>CTA Travel Assistant</h1>"
            "<p>Ask about fares, passes, Ventra cards, accessibility, bikes, and more.</p>"
        )

    # Suggested question chips
    gr.Markdown("**Try a question:**")
    with gr.Row():
        suggestion_btns = [
            gr.Button(q, elem_classes="suggestion-btn", size="sm")
            for q in SUGGESTED_QUESTIONS
        ]

    # Chat area
    chatbot = gr.Chatbot(label="Conversation", height=420, bubble_full_width=False)

    # Input row
    with gr.Row():
        msg = gr.Textbox(
            placeholder = "e.g., How much does a 7-day CTA pass cost?",
            label       = "",
            scale       = 5,
            show_label  = False,
        )
        send_btn = gr.Button("Send", elem_id="send-btn", scale=1)

    clear_btn = gr.Button("Clear conversation", size="sm")

    # Footer
    gr.Markdown(
        "<small>Powered by " + LLM_MODEL_NAME +
        " | Data from transitchicago.com | "
        "For trip planning: <a href='https://www.transitchicago.com/travel-information/'"
        " target='_blank'>CTA Trip Planner</a></small>"
    )

    # State
    state = gr.State([])

    # Event wiring
    msg.submit(chat, [msg, state], [chatbot, state, msg])
    send_btn.click(chat, [msg, state], [chatbot, state, msg])
    for btn in suggestion_btns:
        btn.click(use_suggestion, inputs=btn, outputs=msg)
    clear_btn.click(lambda: ([], []), outputs=[chatbot, state])


demo.launch(share=True, debug=False)


What each line does in Gradio:

- `.bubble-wrap .bot` and `.bubble-wrap .user` — these are the actual class names Gradio 4.x uses internally, replacing the old .message.bot pattern
- `div[data-testid='bot']` and `div[data-testid='user']` — a fallback selector that targets the rendered HTML by its test ID attribute, which is more stable across Gradio versions than class names
- `#EEF2FA` is a very light blue-gray that reads cleanly against dark text on both light and dark system themes — you can swap it for any light color you want
- `color: #1a1a1a !important` on the bot bubble explicitly forces dark text so it doesn't inherit anything invisible from the theme

## Convert to HTML

In [ ]:
import nbformat

with open("ML2_FinalProject_RagPipeline.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

with open("clean_notebook.ipynb", "w") as f:
    nbformat.write(nb, f)

In [ ]:
!jupyter nbconvert --to html clean_notebook.ipynb

In [ ]:
from google.colab import files
files.download("clean_notebook.html")